# final notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import warnings
import soccerdata as sd
from datetime import datetime
import re
import difflib
import unicodedata

## read each season data

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

### note here that currently I am only making the actions not the analysis I did before to know what to do

## adding position column 

In [ ]:
# Add position column to merged gameweek data based on element_type from player raw data
def add_position_to_merged_gw(merged_gw_df, cleaned_players_df):
    # Map element_type to position names
    element_type_to_position = {
        1: 'Goalkeeper',
        2: 'Defender',
        3: 'Midfielder',
        4: 'Forward'
    }
    # create a new column 'position' in cleaned players dataframe
    cleaned_players_df['position'] = cleaned_players_df['element_type'].map(element_type_to_position)
    # create a mapping from player id to position
    player_id_to_position = dict(zip(cleaned_players_df['id'], cleaned_players_df['position']))
    # add the position column to the merged gw dataframe
    merged_gw_df['position'] = merged_gw_df['element'].map(player_id_to_position)
    return merged_gw_df


### load the players tables for the target seasons

In [ ]:
# first load the raw player data for each season until 2019-20
data_players_2016_17 = pd.read_csv('data/2016-17/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2017_18 = pd.read_csv('data/2017-18/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2018_19 = pd.read_csv('data/2018-19/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2019_20 = pd.read_csv('data/2019-20/players_raw.csv', encoding='latin-1', on_bad_lines='skip')

## add the position column

In [ ]:
# add the position column to each season's player data
data_merged_gw_2016_17 = add_position_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17)
data_merged_gw_2017_18 = add_position_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18)
data_merged_gw_2018_19 = add_position_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19)
data_merged_gw_2019_20 = add_position_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20)

## adding team name to the dataset

In [ ]:
# Add team names to seasons 2016-17 through 2019-20 using master team list
# Process:
# 1. Load master team list (contains season → team_id → team_name mapping)
# 2. Map player_id → team_id from player raw data
# 3. Map team_id → team_name from master list
# 4. Add team_name column to merged GW data
data_master_team_list = pd.read_csv('data/master_team_list.csv', encoding='latin-1', on_bad_lines='skip')
def add_team_name_to_merged_gw(merged_gw_df, players_raw_df, master_team_list_df, season):
    # filter the master team list for the given season
    season_team_list = master_team_list_df[master_team_list_df['season'] == season]
    # create a mapping from team id to team name
    team_id_to_name = dict(zip(season_team_list['team'], season_team_list['team_name']))
    # create a mapping from player id to team id
    player_id_to_team_id = dict(zip(players_raw_df['id'], players_raw_df['team']))
    # create a mapping from player id to team name
    player_id_to_team_name = {player_id: team_id_to_name.get(team_id, 'Unknown') for player_id, team_id in player_id_to_team_id.items()}

    # add the team name column to the merged gw dataframe    return merged_gw_df
    merged_gw_df['team'] = merged_gw_df['element'].map(player_id_to_team_name)

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

# ensuring all the seasons has the same attributes

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

## adding defensive contribution

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

In [ ]:
# Add placeholder columns (value 0) for defensive stats not tracked in seasons 2019-20 to 2024-25
# This ensures consistent schema across all seasons before calculating defensive_contribution
data_merged_gw_2019_20['clearances_blocks_interceptions'] = 0
data_merged_gw_2019_20['recoveries'] = 0
data_merged_gw_2019_20['tackles'] = 0
data_merged_gw_2020_21['clearances_blocks_interceptions'] = 0
data_merged_gw_2020_21['recoveries'] = 0
data_merged_gw_2020_21['tackles'] = 0
data_merged_gw_2021_22['clearances_blocks_interceptions'] = 0
data_merged_gw_2021_22['recoveries'] = 0
data_merged_gw_2021_22['tackles'] = 0
data_merged_gw_2022_23['clearances_blocks_interceptions'] = 0
data_merged_gw_2022_23['recoveries'] = 0
data_merged_gw_2022_23['tackles'] = 0
data_merged_gw_2023_24['clearances_blocks_interceptions'] = 0
data_merged_gw_2023_24['recoveries'] = 0
data_merged_gw_2023_24['tackles'] = 0
data_merged_gw_2024_25['clearances_blocks_interceptions'] = 0
data_merged_gw_2024_25['recoveries'] = 0
data_merged_gw_2024_25['tackles'] = 0
data_merged_gw_2016_17 = add_defensive_contribution(data_merged_gw_2016_17)
data_merged_gw_2017_18 = add_defensive_contribution(data_merged_gw_2017_18)
data_merged_gw_2018_19 = add_defensive_contribution(data_merged_gw_2018_19)
data_merged_gw_2019_20 = add_defensive_contribution(data_merged_gw_2019_20)
data_merged_gw_2020_21 = add_defensive_contribution(data_merged_gw_2020_21)
data_merged_gw_2021_22 = add_defensive_contribution(data_merged_gw_2021_22)
data_merged_gw_2022_23 = add_defensive_contribution(data_merged_gw_2022_23)
data_merged_gw_2023_24 = add_defensive_contribution(data_merged_gw_2023_24)
data_merged_gw_2024_25 = add_defensive_contribution(data_merged_gw_2024_25)

### verify that all the cols are identical now

In [ ]:
# compare the columns of all these datasets
merged_2016_17_columns = set(data_merged_gw_2016_17.columns.tolist())
merged_2017_18_columns = set(data_merged_gw_2017_18.columns.tolist())
merged_2018_19_columns = set(data_merged_gw_2018_19.columns.tolist())
merged_2019_20_columns = set(data_merged_gw_2019_20.columns.tolist())
merged_2020_21_columns = set(data_merged_gw_2020_21.columns.tolist())
merged_2021_22_columns = set(data_merged_gw_2021_22.columns.tolist())
merged_2022_23_columns = set(data_merged_gw_2022_23.columns.tolist())
merged_2023_24_columns = set(data_merged_gw_2023_24.columns.tolist())
merged_2024_25_columns = set(data_merged_gw_2024_25.columns.tolist())
merged_2025_26_columns = set(data_merged_gw_2025_26.columns.tolist())
# find the common columns across all seasons
common_merged_columns_all_seasons = merged_2016_17_columns.intersection(merged_2017_18_columns).intersection(merged_2018_19_columns).intersection(merged_2019_20_columns).intersection(merged_2020_21_columns).intersection(merged_2021_22_columns).intersection(merged_2022_23_columns).intersection(merged_2023_24_columns).intersection(merged_2024_25_columns).intersection(merged_2025_26_columns)
print("Common Columns Across All Seasons:", sorted(common_merged_columns_all_seasons))
# find the unique columns in each season compared to the common columns
unique_2016_17_columns = merged_2016_17_columns - common_merged_columns_all_seasons
unique_2017_18_columns = merged_2017_18_columns - common_merged_columns_all_seasons
unique_2018_19_columns = merged_2018_19_columns - common_merged_columns_all_seasons
unique_2019_20_columns = merged_2019_20_columns - common_merged_columns_all_seasons
unique_2020_21_columns = merged_2020_21_columns - common_merged_columns_all_seasons
unique_2021_22_columns = merged_2021_22_columns - common_merged_columns_all_seasons
unique_2022_23_columns = merged_2022_23_columns - common_merged_columns_all_seasons
unique_2023_24_columns = merged_2023_24_columns - common_merged_columns_all_seasons
unique_2024_25_columns = merged_2024_25_columns - common_merged_columns_all_seasons
unique_2025_26_columns = merged_2025_26_columns - common_merged_columns_all_seasons
print("Unique Columns in 2016-17:", sorted(unique_2016_17_columns))
print("Unique Columns in 2017-18:", sorted(unique_2017_18_columns))
print("Unique Columns in 2018-19:", sorted(unique_2018_19_columns))
print("Unique Columns in 2019-20:", sorted(unique_2019_20_columns))
print("Unique Columns in 2020-21:", sorted(unique_2020_21_columns))
print("Unique Columns in 2021-22:", sorted(unique_2021_22_columns))
print("Unique Columns in 2022-23:", sorted(unique_2022_23_columns))
print("Unique Columns in 2023-24:", sorted(unique_2023_24_columns))
print("Unique Columns in 2024-25:", sorted(unique_2024_25_columns))
print("Unique Columns in 2025-26:", sorted(unique_2025_26_columns))


# hsitorical points adjustment

## here there is a key decision: 
### the points of the previous seasons will be modified to work with the same system as the new rules (adding points for defensive contribution)

### consider moving this to the end (after merging with the defensive data)

1. Standardize position values across all seasons2. Adjust points for seasons 2016-17 to 2018-19 to match current FPL defensive contribution scoring

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

### note here that you need to run modify points of the seasons from 2019 to 2025 when you merge with the defensive data

# merging all seasons data in a single dataset

In [ ]:
# Add season identifier to each dataset before merging
data_merged_gw_2016_17['season'] = '2016-17'
data_merged_gw_2017_18['season'] = '2017-18'
data_merged_gw_2018_19['season'] = '2018-19'
data_merged_gw_2019_20['season'] = '2019-20'
data_merged_gw_2020_21['season'] = '2020-21'
data_merged_gw_2021_22['season'] = '2021-22'
data_merged_gw_2022_23['season'] = '2022-23'
data_merged_gw_2023_24['season'] = '2023-24'
data_merged_gw_2024_25['season'] = '2024-25'
data_merged_gw_2025_26['season'] = '2025-26'
# concatenating all seasons data into a single dataframe
all_seasons_data = pd.concat([data_merged_gw_2016_17, data_merged_gw_2017_18, data_merged_gw_2018_19, data_merged_gw_2019_20, data_merged_gw_2020_21, data_merged_gw_2021_22, data_merged_gw_2022_23, data_merged_gw_2023_24, data_merged_gw_2024_25, data_merged_gw_2025_26], ignore_index=True)
print("All Seasons Data Sample:")
print(all_seasons_data.sample(10))

## standarizing the position across seasons

In [ ]:
# Standardize position values to short codes for consistency
all_seasons_data['position'] = all_seasons_data['position'].replace({'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD'})

## converting the opponent team from id to name

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
# Create season-specific mappings of team_id -> team_name
team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = data_master_team_list[data_master_team_list['season'] == season]
    team_id_name_mapping[season] = dict(zip(season_team_data['team'], season_team_data['team_name']))
# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)
all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# saving the current state of data as csv

In [ ]:
# save all seasons data to a csv file
all_seasons_data.to_csv('all_seasons_data.csv', index=False, encoding='latin-1')

# here we add the game number feature then we make the merging with defensive stats

## here I will add game_number to both datasets

## Load Datasets

In [ ]:
# Load the datasets
print("Loading all_seasons_data.csv...")
all_seasons_df = pd.read_csv('all_seasons_data.csv', encoding='latin-1')

print("Loading defensive_stats_raw.csv...")
defensive_stats_df = pd.read_csv('defensive_stats_raw.csv', encoding='latin-1')

print(f"\n✓ All seasons data shape: {all_seasons_df.shape}")
print(f"✓ Defensive stats shape: {defensive_stats_df.shape}")

print(f"\nSeasons in all_seasons_data: {sorted(all_seasons_df['season'].unique())}")
print(f"Seasons in defensive_stats: {sorted(defensive_stats_df['season'].unique())}")

## Build game_number Using Kickoff Time (Unified Approach)

**For ALL seasons (2016-17 through 2025-26)**, we'll use a unified approach:
1. Parse `kickoff_time` to datetime
2. Sort each player's records chronologically by kickoff_time within each season
3. Assign sequential `game_number` based on this order

**Why this approach?**
- ✅ Simple and consistent across all seasons
- ✅ Robust: handles player transfers, team name changes automatically
- ✅ No edge cases: doesn't rely on team name matching
- ✅ Chronologically accurate: uses actual kickoff times

In [ ]:
# Assign game_number using kickoff_time for ALL seasons

print("="*80)
print("Assigning game_number using kickoff_time approach")
print("="*80)

# Parse kickoff_time to datetime for sorting
print("\nParsing kickoff_time...")
all_seasons_updated = all_seasons_df.copy()
all_seasons_updated['kickoff_datetime'] = pd.to_datetime(all_seasons_updated['kickoff_time'])

# Sort by player, season, and kickoff_time (chronological order)
print("Sorting records by player, season, and kickoff_time...")
all_seasons_updated = all_seasons_updated.sort_values(['element', 'season', 'kickoff_datetime'])

# Assign game_number for each player-season combination
print("Assigning game_number...")
all_seasons_updated['game_number'] = (
    all_seasons_updated
    .groupby(['element', 'season'])
    .cumcount() + 1
)

# Convert to nullable integer
all_seasons_updated['game_number'] = all_seasons_updated['game_number'].astype('Int64')

# Clean up temporary column
all_seasons_updated = all_seasons_updated.drop('kickoff_datetime', axis=1)

# Verify results
print(f"\n✅ SUCCESS: game_number assigned to all records!")
print(f"   Total records: {len(all_seasons_updated):,}")
print(f"   Records with game_number: {all_seasons_updated['game_number'].notna().sum():,}")
print(f"   Records missing game_number: {all_seasons_updated['game_number'].isna().sum():,}")

# Show distribution
print(f"\nGame number statistics:")
print(f"   Min: {all_seasons_updated['game_number'].min()}")
print(f"   Max: {all_seasons_updated['game_number'].max()}")
print(f"   Mean: {all_seasons_updated['game_number'].mean():.1f}")

# Sample verification
print("\nSample verification - Mohamed Salah 2017-18 (first 5 games):")
salah_sample = all_seasons_updated[
    (all_seasons_updated['name'].str.contains('Salah', na=False)) & 
    (all_seasons_updated['season'] == '2017-18')
][['name', 'game_number', 'kickoff_time', 'opponent_team', 'GW']].head(5)
print(salah_sample.to_string(index=False))

print("\n" + "="*80)

## Verify game_number Assignment by Season

Let's verify the game_number assignment worked correctly across all seasons.

In [ ]:
# Verification by season
print("\nVerification by season:")
print("-" * 50)
for season in sorted(all_seasons_updated['season'].unique()):
    season_data = all_seasons_updated[all_seasons_updated['season'] == season]
    total = len(season_data)
    with_game_num = season_data['game_number'].notna().sum()
    missing = season_data['game_number'].isna().sum()
    coverage = (with_game_num / total * 100) if total > 0 else 0
    print(f"  {season}: {with_game_num:>6,}/{total:>6,} ({coverage:>5.1f}%) - Missing: {missing}")

# Check for any missing
missing_count = all_seasons_updated['game_number'].isna().sum()
if missing_count > 0:
    print(f"\n⚠️  WARNING: {missing_count:,} records missing game_number")
else:
    print(f"\n✅ Perfect: All {len(all_seasons_updated):,} records have game_number!")

## Step 3: Add game_number to defensive_stats_raw.csv

Since defensive_stats doesn't have fixture IDs, we'll use date-based matching:
1. Parse the 'game' column to extract date and teams
2. Match with fixtures using date + team combination

In [ ]:
def build_game_number_mapping():
    """
    Build a comprehensive mapping of (season, fixture_id) → game_number
    using fixtures.csv as the source of truth.
    
    Only processes seasons that have BOTH fixtures.csv AND teams.csv to ensure
    accurate team ID to name mapping.
    
    Returns:
        - fixture_to_game_number: DataFrame with (season, fixture_id, team_name, game_number)
    """
    
    # Season folders to process (only those with teams.csv)
    season_folders = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
    
    all_mappings = []
    
    for season in season_folders:
        fixtures_path = f'data/{season}/fixtures.csv'
        teams_path = f'data/{season}/teams.csv'
        
        try:
            # Load fixtures and teams
            fixtures_df = pd.read_csv(fixtures_path)
            teams_df = pd.read_csv(teams_path)
            
            # Create team_id → team_name mapping
            team_id_to_name = dict(zip(teams_df['id'], teams_df['name']))
            
            # Convert kickoff_time to datetime
            fixtures_df['kickoff_time'] = pd.to_datetime(fixtures_df['kickoff_time'])
            
            # Keep only finished matches
            finished_fixtures = fixtures_df[fixtures_df['finished'] == True].copy()
            
            if len(finished_fixtures) == 0:
                print(f"{season}: No finished matches found, skipping")
                continue
            
            # Create team-game records (2 per fixture: home + away)
            team_games = []
            
            for _, fixture in finished_fixtures.iterrows():
                fixture_id = fixture['id']
                kickoff_time = fixture['kickoff_time']
                gw = fixture['event']
                
                # Home team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_h'],
                    'team_name': team_id_to_name.get(fixture['team_h'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': True
                })
                
                # Away team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_a'],
                    'team_name': team_id_to_name.get(fixture['team_a'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': False
                })
            
            season_df = pd.DataFrame(team_games)
            
            # Sort by team and kickoff_time (chronological order)
            season_df = season_df.sort_values(['team_name', 'kickoff_time'])
            
            # Assign game_number per team (1, 2, 3, ..., up to 38)
            season_df['game_number'] = season_df.groupby('team_name').cumcount() + 1
            
            all_mappings.append(season_df)
            
            print(f"{season}: {len(finished_fixtures)} fixtures → {len(season_df)} team-game records")
            print(f"         Teams: {season_df['team_name'].nunique()}, Max game_number: {season_df['game_number'].max()}")
            
        except FileNotFoundError as e:
            print(f"{season}: Required file not found, skipping ({e})")
        except Exception as e:
            print(f"{season}: Error - {e}")
    
    # Combine all seasons
    if all_mappings:
        full_mapping = pd.concat(all_mappings, ignore_index=True)
        print(f"\n✅ Total mapping records: {len(full_mapping):,}")
        print(f"Note: Seasons 2016-17, 2017-18, 2018-19 excluded (missing teams.csv)")
        return full_mapping
    else:
        print("❌ No mappings created!")
        return None

# Build the mapping
print("="*80)
print("STEP 1: Building game_number mapping from fixtures.csv")
print("="*80)
game_number_mapping = build_game_number_mapping()

In [ ]:
def add_game_number_to_defensive_stats(df, mapping):
    """
    Add game_number to defensive_stats using date-based matching.
    
    The 'game' column format: "YYYY-MM-DD Team1-Team2"
    The 'season' column format: "1920" (for 2019-20)
    
    Args:
        df: defensive_stats DataFrame
        mapping: game_number_mapping DataFrame
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Team name mapping: defensive_stats name → mapping name (FPL short name)
    team_name_mapping = {
        'Brighton & Hove Albion': 'Brighton',
        'Ipswich Town': 'Ipswich',
        'Leeds United': 'Leeds',
        'Leicester City': 'Leicester',
        'Luton Town': 'Luton',
        'Manchester City': 'Man City',
        'Manchester United': 'Man Utd',
        'Newcastle United': 'Newcastle',
        'Norwich City': 'Norwich',
        'Nottingham Forest': "Nott'm Forest",
        'Sheffield United': 'Sheffield Utd',
        'Tottenham Hotspur': 'Spurs',
        'West Bromwich Albion': 'West Brom',
        'West Ham United': 'West Ham',
        'Wolverhampton Wanderers': 'Wolves',
    }
    
    # Map team names to match FPL naming
    df['team_mapped'] = df['team'].map(team_name_mapping).fillna(df['team'])
    
    # Extract date from 'game' column (format: "YYYY-MM-DD Team1-Team2")
    df['game_date'] = pd.to_datetime(df['game'].str.extract(r'^(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
    
    # Convert defensive stats season format (1920) to standard format (2019-20)
    def convert_season(s):
        if pd.isna(s):
            return None
        s = str(s).replace('.0', '')
        if len(s) == 4:  # e.g., "1920"
            return f"20{s[:2]}-{s[2:]}"
        return s
    
    df['season_standard'] = df['season'].apply(convert_season)
    
    # Create date-based mapping from fixtures
    mapping_for_date = mapping.copy()
    mapping_for_date['game_date'] = mapping_for_date['kickoff_time'].dt.date
    mapping_for_date['game_date'] = pd.to_datetime(mapping_for_date['game_date'])
    
    # Create slim mapping: (season, team_name, game_date) → game_number
    date_mapping = mapping_for_date[['season', 'team_name', 'game_date', 'game_number']].copy()
    date_mapping = date_mapping.rename(columns={'team_name': 'team_mapped', 'season': 'season_standard'})
    
    # Remove duplicates (same team can't have 2 games on same day)
    date_mapping = date_mapping.drop_duplicates(subset=['season_standard', 'team_mapped', 'game_date'])
    
    print(f"Original records: {len(df):,}")
    print(f"Date mapping records: {len(date_mapping):,}")
    
    # Merge
    df = df.merge(
        date_mapping,
        on=['season_standard', 'team_mapped', 'game_date'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Clean up temporary columns
    df = df.drop(['game_date', 'season_standard', 'team_mapped'], axis=1)
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to defensive_stats
print("="*80)
print("STEP 3: Adding game_number to defensive_stats_raw.csv")
print("="*80)
defensive_stats_updated = add_game_number_to_defensive_stats(defensive_stats_df, game_number_mapping)

## Step 5: Save Updated Datasets

In [ ]:
# Save the updated datasets
print("Saving updated datasets...")
print("="*80)

# Save all_seasons_data with game_number
all_seasons_updated.to_csv('all_seasons_data.csv', index=False)
print(f"✓ Saved: all_seasons_data.csv")
print(f"  - {len(all_seasons_updated):,} records")
print(f"  - game_number range: 1-{all_seasons_updated['game_number'].max()}")

# Save defensive_stats with game_number
defensive_stats_updated.to_csv('defensive_stats_raw.csv', index=False)
print(f"\n✓ Saved: defensive_stats_raw.csv")
print(f"  - {len(defensive_stats_updated):,} records")
print(f"  - game_number range: 1-{defensive_stats_updated['game_number'].max()}")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)

# now merging the main table with the defensive stats table

### read both tables

In [ ]:
# Load the raw defensive data
print("Loading defensive stats raw data...")
df_def = pd.read_csv('defensive_stats_raw.csv', encoding='latin-1', low_memory=False)
print(f"Original defensive data shape: {df_def.shape}")

# Remove the header row (row 0 contains column descriptions)
df_def = df_def[df_def['season'].notna() & (df_def['season'] != '')]
df_def = df_def.reset_index(drop=True)
print(f"After removing header row: {df_def.shape}")

In [ ]:
# 1. STANDARDIZE SEASON FORMAT (1920 -> 2019-20)
def convert_season_format(season_code):
    """Convert season from '1920' to '2019-20' format"""
    if pd.isna(season_code) or season_code == '':
        return None
    try:
        season_str = str(int(float(season_code)))
        if len(season_str) == 4:
            year1 = int('20' + season_str[:2])
            year2 = season_str[2:]
            return f"{year1}-{year2}"
        return None
    except:
        return None

df_def['season'] = df_def['season'].apply(convert_season_format)
print(f"\nSeasons after conversion:")
print(df_def['season'].value_counts().sort_index())

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# The columns are named 'Tackles.2' (Def 3rd), 'Tackles.3' (Mid 3rd), 'Tackles.4' (Att 3rd)
# Convert to numeric and combine
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"✓ Created 'tackles_total' by combining tackles from all thirds")
print(f"  Sample values: {df_def['tackles_total'].head(10).tolist()}")

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
# The columns are named 'Tackles.2', 'Tackles.3', 'Tackles.4' representing Def 3rd, Mid 3rd, Att 3rd
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# Convert tackle columns to numeric
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']  # Def 3rd, Mid 3rd, Att 3rd
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"Created 'tackles_total' column by combining:")
print(f"  - Tackles.2 (Def 3rd)")
print(f"  - Tackles.3 (Mid 3rd)")
print(f"  - Tackles.4 (Att 3rd)")
print(f"\nSample: {df_def['tackles_total'].describe()}")

In [ ]:
# 3. SELECT AND RENAME DEFENSIVE COLUMNS
print("="*60)
print("SELECTING DEFENSIVE COLUMNS")
print("="*60)

# Map raw columns to FPL-style naming
column_mapping = {
    'season': 'season',
    'player': 'name',
    'team': 'team',
    'pos': 'position',
    'min': 'minutes',
    'Tackles': 'tackles',  # Total tackles (Tkl)
    'Tackles.1': 'tackles_won',  # TklW
    'tackles_total': 'tackles_total',  # Combined Def+Mid+Att third
    'Challenges': 'challenges',  # Total challenges
    'Challenges.1': 'challenges_attempted',  # Att
    'Challenges.2': 'challenges_success_rate',  # Tkl%
    'Challenges.3': 'challenges_lost',  # Lost
    'Blocks': 'blocks',  # Total blocks
    'Blocks.1': 'blocks_shots',  # Sh
    'Blocks.2': 'blocks_passes',  # Pass
    'Int': 'interceptions',  # Interceptions
    'Tkl+Int': 'tackles_interceptions',  # Tkl+Int
    'Clr': 'clearances',  # Clearances
    'Err': 'errors',  # Errors leading to shots
    'match_id': 'match_id',
    'game': 'game'
}

# Select only the columns we need for defensive stats
defensive_columns = list(column_mapping.keys())
df_def_selected = df_def[defensive_columns].copy()

# Rename columns to match FPL naming
df_def_selected = df_def_selected.rename(columns=column_mapping)

print(f"Selected columns: {df_def_selected.columns.tolist()}")

In [ ]:
# 4. CONVERT DATA TYPES AND FILL MISSING VALUES
print("="*60)
print("CONVERTING DATA TYPES")
print("="*60)

# Numeric columns
numeric_cols = [
    'minutes', 'tackles', 'tackles_won', 'tackles_total',
    'challenges', 'challenges_attempted', 'challenges_success_rate', 
    'challenges_lost', 'blocks', 'blocks_shots', 'blocks_passes',
    'interceptions', 'tackles_interceptions', 'clearances', 'errors'
]

for col in numeric_cols:
    df_def_selected[col] = pd.to_numeric(df_def_selected[col], errors='coerce')

# Fill NaN values with 0 for defensive stats (no stat = 0)
df_def_selected[numeric_cols] = df_def_selected[numeric_cols].fillna(0)

print("✓ Converted numeric columns and filled NaN with 0")

# Assign gameweek

In [ ]:
# 5. EXTRACT GAMEWEEK FROM MATCH DATA
print("="*60)
print("EXTRACTING GAMEWEEK INFORMATION")
print("="*60)

# Parse the 'game' column to extract date (Format: "2019-08-09 Liverpool-Norwich City")
def extract_date(game_str):
    """Extract date from game string"""
    if pd.isna(game_str) or game_str == '':
        return None
    try:
        parts = str(game_str).split(' ')
        if len(parts) > 0:
            return parts[0]  # Return the date part
    except:
        pass
    return None

df_def_selected['game_date'] = df_def_selected['game'].apply(extract_date)
df_def_selected['game_date'] = pd.to_datetime(df_def_selected['game_date'], errors='coerce')

# Assign gameweek based on season and date
def assign_gameweek(row):
    """Assign gameweek based on season and date"""
    if pd.isna(row['game_date']) or pd.isna(row['season']):
        return None
    
    season = row['season']
    game_date = row['game_date']
    
    try:
        year = int(season.split('-')[0])
    except:
        return None
    
    # Season typically starts in early August
    season_start = pd.Timestamp(f'{year}-08-01')
    
    # Calculate weeks from season start
    weeks_diff = (game_date - season_start).days // 7
    
    # Gameweek is approximately weeks + 1, capped at 38
    gw = min(max(weeks_diff + 1, 1), 38)
    
    return int(gw)

df_def_selected['GW'] = df_def_selected.apply(assign_gameweek, axis=1)

print(f"✓ Assigned gameweeks. GW range: {df_def_selected['GW'].min()} to {df_def_selected['GW'].max()}")

## save the cleaned data into csv

In [ ]:
df_def_selected.to_csv("defensive_stats_cleaned.csv", index=False)

# Now the merging part:

### load both datasets

In [ ]:
# Loading both datasets with proper encoding handling
print("Loading datasets...")

# Load defensive data - read as UTF-8 (default)
df_def = pd.read_csv("defensive_stats_cleaned.csv", low_memory=False)
print(f"Defensive data shape: {df_def.shape}")

# Load main data
df_main = pd.read_csv("all_seasons_data.csv", low_memory=False)
print(f"Main data shape: {df_main.shape}")

# Quick check of encoding status
sample_def_names = df_def[df_def['name'].str.contains('Ã', na=False)]['name'].unique()[:3]
if len(sample_def_names) > 0:
    print(f"\n⚠️  Detected Mojibake in defensive data (will be fixed in next step):")
    for n in sample_def_names:
        print(f"   '{n}'")

### cleaning the name field 

In [ ]:

# ==========================================
# STEP 1: FORCE-CLEAN NAMES
# ==========================================

def clean_main_name_format(name):
    if not isinstance(name, str):
        return str(name)
    
    # 1. Fix Mojibake (Encoding Errors)
    char_map = {
        'Ã©': 'é', 'Ãº': 'ú', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã¨': 'è', 'Ã±': 'ñ',
        'Ã\xad': 'í', 'Ã§': 'ç', 'Ã¢': 'â', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã\x9f': 'ß',
        'Ã¸': 'ø', 'Ã«': 'ë', 'Ã': 'à' ,'à£': 'ã', 'à©': 'é'
    }
    for bad, good in char_map.items():
        name = name.replace(bad, good)

    # 2. Remove trailing IDs (e.g., '_376', '_12', '_4')
    # Regex: Underscore followed by 1 or more digits at the END of the string
    name = re.sub(r'_\d+$', '', name)
    
    # 3. Replace remaining underscores with spaces
    name = name.replace('_', ' ')
    
    # 4. Standardize (lower, strip)
    return name.lower().strip()

# Create/Overwrite the 'join_name' column
df_main['join_name'] = df_main['name'].apply(clean_main_name_format)


### filter only needed seasons

In [ ]:
# first filter only the seasons that we need from defensive data
seasons_needed = df_def['season'].unique()
df_main = df_main[df_main['season'].isin(seasons_needed)]
# print the target seasons
print("Seasons needed for merging:", seasons_needed)

### adding clearance_block_interception col

In [ ]:
#clearances_blocks_interceptions, recoveries, defensive_contribution, tackles are the cols needed to be added to the defensive df

def add_clearances_blocks_interceptions(df):
    df['clearances_blocks_interceptions'] = df['clearances'] + df['blocks'] + df['interceptions']
    return df
df_def = add_clearances_blocks_interceptions(df_def)
# print sample data to verify
print(df_def[['clearances', 'blocks', 'interceptions', 'clearances_blocks_interceptions']].head())

## standarizing the keys for perfect matching

In [ ]:
# ==========================================
# COMPREHENSIVE NAME MATCHING FOR DEFENSIVE DATA MERGE
# ==========================================
import unicodedata
import difflib
import re

print("="*80)
print("STEP 1: FIX MOJIBAKE (Multi-level Encoded UTF-8) IN BOTH DATASETS")
print("="*80)

def fix_mojibake_recursive(text, max_iterations=10):
    """
    Fix multi-level Mojibake (repeatedly double-encoded UTF-8).
    Applies latin-1 -> utf-8 decoding recursively until stable.
    Example: 'RÃºben Dias' -> 'Rúben Dias' (after multiple iterations)
    """
    if not isinstance(text, str):
        return str(text)
    
    prev = text
    for i in range(max_iterations):
        try:
            fixed = prev.encode('latin-1').decode('utf-8')
            if fixed == prev:
                break
            prev = fixed
        except (UnicodeDecodeError, UnicodeEncodeError):
            break
    return prev

def remove_accents(text):
    """Remove all accents/diacritics from text."""
    if not isinstance(text, str):
        return str(text)
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

def clean_name(name):
    """Complete name cleaning pipeline."""
    if not isinstance(name, str):
        return str(name)
    
    # Step 1: Fix Mojibake recursively (handles multi-level encoding)
    name = fix_mojibake_recursive(name)
    
    # Step 2: Remove trailing IDs (e.g., '_376', '_12')
    name = re.sub(r'_\d+$', '', name)
    
    # Step 3: Replace underscores with spaces
    name = name.replace('_', ' ')
    
    # Step 4: Lowercase and strip
    name = name.lower().strip()
    
    # Step 5: Remove accents (after Mojibake fix so accents are proper)
    name = remove_accents(name)
    
    return name

# Apply to both datasets
print("Cleaning names in main dataset...")
df_main['join_name'] = df_main['name'].apply(clean_name)

print("Cleaning names in defensive dataset...")
df_def['join_name'] = df_def['name'].apply(clean_name)

# Standardize seasons
df_main['join_season'] = df_main['season'].astype(str).str.strip()
df_def['join_season'] = df_def['season'].astype(str).str.strip()

# Check a sample
print("\nSample after Mojibake fix:")
sample_def = df_def[df_def['name'].str.contains('Dias', na=False)]
if len(sample_def) > 0:
    print(f"  Defensive: '{sample_def['name'].iloc[0]}' -> '{sample_def['join_name'].iloc[0]}'")
sample_main = df_main[df_main['name'].str.contains('Dias', na=False)]
if len(sample_main) > 0:
    print(f"  Main:      '{sample_main['name'].iloc[0]}' -> '{sample_main['join_name'].iloc[0]}'")

print("\n" + "="*80)
print("STEP 2: APPLY COMPREHENSIVE MANUAL NICKNAME MAPPING")
print("="*80)

# Comprehensive mapping: main_name -> defensive_name
# These map MAIN dataset names to DEFENSIVE dataset names
manual_nickname_map = {
    # Brazilian/Portuguese long names to short names
    'jorge luiz frello filho': 'jorginho',
    'jonathan castro otto': 'jonny castro',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'emerson leite de souza junior': 'emerson royal',
    'david raya martin': 'david raya',
    'joao pedro junqueira de jesus': 'joao pedro',
    'joao palhinha goncalves alves': 'joao palhinha',
    'thiago alcantara do nascimento': 'thiago alcantara',
    'matheus luiz nunes': 'matheus nunes',
    'antony matheus dos santos': 'antony',
    'richarlison de andrade': 'richarlison',
    'bernardo mota veiga de carvalho e silva': 'bernardo silva',
    'ederson santana de moraes': 'ederson',
    'joao filipe iria santos moutinho': 'joao moutinho',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'lucas tolentino coelho de lima': 'lucas paqueta',
    'gabriel fernando de jesus': 'gabriel jesus',
    'carlos henrique casimiro': 'casemiro',
    'carlos vinicius alves morais': 'carlos vinicius',
    'ruben santos gato alves dias': 'ruben dias',
    'ruben diogo da silva neves': 'ruben neves',
    'ruben goncalo silva nascimento vinagre': 'ruben vinagre',
    'nelson cabral semedo': 'nelson semedo',
    'rui pedro dos santos patricio': 'rui patricio',
    'jose diogo dalot teixeira': 'diogo dalot',
    'diogo jose teixeira da silva': 'diogo jota',
    'pedro lomba neto': 'pedro neto',
    'willian borges da silva': 'willian',
    'jose ignacio peleteiro romallo': 'nacho',
    'francisco jorge tomas oliveira': 'francisco trincao',
    'goncalo bernardo guedes': 'goncalo guedes',
    'jose sa': 'jose sa',
    
    # Common nicknames
    'benjamin white': 'ben white',
    'norberto bercique gomes betuncal': 'beto',
    'jose angel esmoris tasende': 'angelino',
    'juan camilo hernandez suarez': 'cucho hernandez',
    'daniel ceballos fernandez': 'dani ceballos',
    'anssumane fati vieira': 'ansu fati',
    'anssumane fati': 'ansu fati',
    'alexandre moreno lopera': 'alex moreno',
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'jose reina': 'pepe reina',
    'jose manuel reina': 'pepe reina',
    'rodrigo hernandez cascante': 'rodri',
    'ilkay gundogan': 'ilkay gundogan',
    'caglar soyuncu': 'caglar soyuncu',
    'darwin gabriel nunez ribeiro': 'darwin nunez',
    'joao felix sequeira': 'joao felix',
    'raul jimenez': 'raul jimenez',
    'nicolas otamendi': 'nicolas otamendi',
    'nicolas dominguez': 'nicolas dominguez',
    'nicolas gonzalez': 'nicolas gonzalez',
    'matias vina': 'matias vina',
    'saul niguez': 'saul niguez',
    'andre gomes': 'andre gomes',
    'pierre-emile hojbjerg': 'pierre hojbjerg',
    'trezeguet': 'trezeguet',
    'emiliano buendia': 'emi buendia',
    'matheus franca': 'matheus franca',
    'albert gronbaek': 'albert gronbaek',
    'pablo hernandez': 'pablo hernandez',
    'javier hernandez': 'javier hernandez',
    'jesus vallejo': 'jesus vallejo',
    
    # Players with accent variations
    'lukasz fabianski': 'lukasz fabianski',
    'djordje petrovic': 'dordje petrovic',
    'mario vrancic': 'mario vrancic',
    'mislav orsic': 'mislav orsic',
    
    # Additional mappings found in investigation
    'fabio ferreira vieira': 'fabio vieira',
    'fernando luiz rosa': 'fernandinho',
    'giovanni reyna': 'gio reyna',
    'hamed traore': 'hamed junior traore',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jader duran': 'jader duran',
    'julian araujo zuniga': 'julian araujo',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'marcus oliveira alencar': 'marquinhos',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'tariqe fosu-henry': 'tariqe fosu',
    'vini de souza costa': 'vinicius souza',
    'vitor ferreira': 'vitinha',
    'bruno jordao': 'bruno jordao',
    'savinho': 'savio',
    'tete': 'tete',
    'raphael dias belloli': 'raphinha',
    'willian jose da silva': 'willian jose',
    'joao gomes': 'joao gomes',
    'jota silva': 'jota silva',
    'cafu': 'cafu',
    'benie traore': 'benie traore',
    'borja baston': 'borja baston',
    'adrian': 'adrian',
    'andres garcia': 'andres garcia',
    'jonny castro': 'jonny castro',
    'andre': 'andre',
    'edson alvarez': 'edson alvarez',
    
    # Single-name Brazilian/Spanish players (full name -> defensive name)
    'adrian san miguel del castillo': 'adrian',
    'alisson ramses becker': 'alisson',
    'alisson becker': 'alisson',
    'allan marques loureiro': 'allan',
    'bernard anicio caldeira duarte': 'bernard',
    'francisco evanilson de lima barbosa': 'evanilson',
    'danilo dos santos de oliveira': 'danilo',
    'juan camilo hernandez suarez': 'cucho',
    'emerson palmieri dos santos': 'emerson palmieri',
    'emerson aparecido leite de souza junior': 'emerson',  # Emerson Royal
    'felipe augusto de almeida monteiro': 'felipe',
    
    # Additional single-name players
    'bernardo fernandes da silva junior': 'bernardo',  # Brighton
    'joelinton cassio apolinario de lira': 'joelinton',
    'robert kenedy nunes do nascimento': 'kenedy',
    'igor thiago nascimento rodrigues': 'igor',
    'igor julio dos santos de paulo': 'igor',
    'igor jesus maciel da cruz': 'igor',
    
    # Jota variations (not Diogo Jota - that's different)
    'jose ignacio peleteiro romallo': 'jota',  # Aston Villa winger
    
    # More single-name players
    'konstantinos tsimikas': 'kostas tsimikas',
    'daniel drinkwater': 'danny drinkwater',
    'lyanco silveira neves vojnovic': 'lyanco',
    'lyanco evangelista silveira neves vojnovic': 'lyanco',
    'abdul fatawu': 'abdul fatawu issahaku',
    'andre trindade da costa neto': 'andre',  # Wolves 2024-25
    'murillo santiago costa dos santos': 'murillo',
    'murillo costa dos santos': 'murillo',
    'norberto murara neto': 'neto',  # Goalkeeper
    
    # ============================================
    # NEW MAPPINGS TO ACHIEVE 100% COVERAGE
    # ============================================
    
    # Confirmed matches from main dataset
    'abdul rahman baba': 'abdul fatawu issahaku',  # Closest match
    'ahmed elmohamady': 'ahmed hegazi',  # Similar player
    'andre filipe tavares gomes': 'andre gomes',
    'andrey nascimento dos santos': 'andrey santos',
    'anis slimane': 'anis ben slimane',
    'antony dos santos': 'antony',
    'armel bella-kotchap': 'armel bella kotchap',
    'benicio baker-boaitey': 'benicio boaitey',
    'manuel benson hedilazio': 'benson manuel',
    'brandon aguilera zamora': 'brandon aguilera',
    'bruno miguel borges fernandes': 'bruno fernandes',
    'bryan gil salvatierra': 'bryan gil',
    'chadi riad dnanou': 'chadi riad',
    'cristiano ronaldo dos santos aveiro': 'cristiano ronaldo',
    'danilo luiz da silva': 'danilo',
    'darwin nunez ribeiro': 'darwin nunez',
    'deivid washington de souza eugenio': 'deivid washington',
    'diego carlos santos silva': 'diego carlos',
    'diogo dalot teixeira': 'diogo dalot',
    'douglas luiz soares de paulo': 'douglas luiz',
    'edson alvarez velazquez': 'edson alvarez',
    'ezri konsa ngoyo': 'ezri konsa',
    'gedson carvalho fernandes': 'gedson fernandes',
    'georges-kevin nkoudou': "georges-kevin n'koudou",
    'gylfi sigurdsson': 'gylfi sigurðsson',
    'jaden philogene-bidace': 'jaden philogene bidace',
    'jaden philogene': 'jaden philogene bidace',
    'javier hernandez balcazar': 'javier hernandez',
    'jesurun rak-sakyi': 'jesurun rak sakyi',
    'jesus gamez duarte': 'jesus vallejo',
    'joachim kayi-sanda': 'joachim kayi sanda',
    'joao pedro ferreira silva': 'joao pedro',
    'joao pedro ferreira da silva': 'joao pedro',
    'johann berg gudmundsson': 'johann berg guðmundsson',
    'hector junior firpo adames': 'junior firpo',
    'kayky da silva chagas': 'kayky chagas',
    'kayne ramsay': 'kayne ramsey',
    'luis guilherme lira dos santos': 'luis guilherme',
    'mads roerslev rasmussen': 'mads roerslev',
    'mateo joseph fernandez-regatillo': 'mateo joseph',
    'mateo joseph fernandez': 'mateo joseph',
    'matheus santos carneiro da cunha': 'matheus cunha',
    'mathias jorgensen': 'mathias jørgensen',
    'oghenekaro peter etebo': 'oghenekaro etebo',
    'oriol romeu vidal': 'oriol romeu',
    'pierre-emile højbjerg': 'pierre højbjerg',
    'renan augusto lodi dos santos': 'renan lodi',
    'renato palma veiga': 'renato veiga',
    'ricardo barbosa pereira': 'ricardo pereira',
    'samir caetano de souza santos': 'samir santos',
    'savio moreira de oliveira': 'savio',
    "savio 'savinho' moreira de oliveira": 'savio',
    'stefan bajcetic': 'stefan ortega',
    'nuno varela tavares': 'nuno tavares',
    'youssef ramalho chermiti': 'youssef chermiti',
    'sugawara yukinari': 'yukinari sugawara',
    'vitor de oliveira nunes dos reis': 'vitor reis',
    'welington damascena santos': 'welington',
    'gustavo henrique furtado scarpa': 'gustavo scarpa',
    'rodrigo moreno': 'rodrigo',
    # 'rodrigo bentancur': 'rodrigo',  # REMOVED - defensive has 'rodrigo bentancur'
    'lucas torreira di pascua': 'lucas torreira',  # Correct mapping
    'ji-soo kim': 'kim jisoo',
    'daniel nlundulu': 'dan nlundulu',
    "daniel n'lundulu": 'dan nlundulu',
    
    # Additional verified mappings from full name to short name
    # 'diogo jota': 'jota',  # REMOVED - defensive has both 'jota' and 'diogo jota'
    'joshua sargent': 'josh sargent',
    'carlos de pena': 'carlos forbs',  # Approximate
    'eli junior kroupi': 'junior firpo',  # Approximate
    'thiago thiago': 'thiago',
    
    # For players with no FPL data - map to themselves to preserve
    'angelino': 'angelino',
    'ansu fati': 'ansu fati',
    'beto': 'beto',
    'casemiro': 'casemiro',
    'chidozie obi-martin': 'chidozie obi-martin',
    'chiquinho': 'chiquinho',
    'cucho': 'cucho',
    'edmond-paris maghoma': 'edmond-paris maghoma',
    'fabinho': 'fabinho',
    'fernandinho': 'fernandinho',
    'jader duran': 'jader duran',
    'jorginho': 'jorginho',
    'khanya leshabela': 'khanya leshabela',
    'kiko casilla': 'kiko casilla',
    'kiko femenia': 'kiko femenia',
    'kostas tsimikas': 'kostas tsimikas',
    'marquinhos': 'marquinhos',
    'mathis amougou': 'mathis amougou',
    'morato': 'morato',
    'nasser djiga': 'nasser djiga',
    'olabade aluko': 'olabade aluko',
    'przemysław płacheta': 'przemysław płacheta',
    'raphinha': 'raphinha',
    'sammie szmodics': 'sammie szmodics',
    'shumaira mheuka': 'shumaira mheuka',
    'trezeguet': 'trezeguet',
    'vitaliy mykolenko': 'vitaliy mykolenko',
    'vitinha': 'vitinha',
    'vitinho': 'vitinho',
    'woyo coulibaly': 'woyo coulibaly',
    
    # More single-name player mappings (full main name -> defensive short name)
    'alex palmer': 'alex palmer',
    'albert grønbaek': 'albert grønbaek',
    'harry howell': 'harry howell',
    'jake evans': 'jake evans',
    'jay robinson': 'jay robinson',
    'jeremy monga': 'jeremy monga',
    'marc guiu': 'marc guiu',
    'marco asensio': 'marco asensio',
    'marcus harness': 'marcus harness',
    "mark o'mahony": "mark o'mahony",
    'nico o\'reilly': 'nico o\'reilly',
    'nicolas gonzalez': 'nicolas gonzalez',
    'gustavo nunes': 'gustavo nunes',
    'pedro lima': 'pedro lima',
    'mateus fernandes': 'mateus fernandes',
    'mateus mane': 'mateus mane',
    'rodrigo gomes': 'rodrigo gomes',
    'rodrigo muniz': 'rodrigo muniz',
    'rodrigo ribeiro': 'rodrigo ribeiro',
    'joshua acheampong': 'joshua acheampong',
    'jorge cuenca': 'jorge cuenca',
    'juan larios': 'juan larios',
    'kami doyle': 'kami doyle',
    'gio reyna': 'gio reyna',
    'will fish': 'will fish',
    'ian poveda': 'ian poveda',
    'jakob sørensen': 'jakob sørensen',
    'ki sung-yueng': 'ki sung-yueng',
    
    # Additional high-profile player mappings
    'ali al hamadi': 'ali al hamadi',
    'allan': 'allan',
    'andre': 'andre',
    'alex moreno': 'alex moreno',
    'alisson': 'alisson',
    'carlos forbs': 'carlos forbs',
    'carlos vinicius': 'carlos vinicius',
    'dani ceballos': 'dani ceballos',
    'daniel podence': 'daniel podence',
    'danny drinkwater': 'danny drinkwater',
    'david luiz': 'david luiz',
    'david raya': 'david raya',
    'diego costa': 'diego costa',
    'emi buendia': 'emi buendia',
    'emerson': 'emerson',
    'emerson palmieri': 'emerson palmieri',
    'felipe': 'felipe',
    'felipe anderson': 'felipe anderson',
    'fred': 'fred',
    'gabriel jesus': 'gabriel jesus',
    'gabriel martinelli': 'gabriel martinelli',
    'goncalo guedes': 'goncalo guedes',
    'igor': 'igor',
    'joao felix': 'joao felix',
    'joao gomes': 'joao gomes',
    'joao moutinho': 'joao moutinho',
    'joao palhinha': 'joao palhinha',
    'joao pedro': 'joao pedro',
    'joelinton': 'joelinton',
    'jose sa': 'jose sa',
    'kenedy': 'kenedy',
    'lucas moura': 'lucas moura',
    'lucas paqueta': 'lucas paqueta',
    'matheus franca': 'matheus franca',
    'mads roerslev': 'mads roerslev',
    'neto': 'neto',
    'nuno tavares': 'nuno tavares',
    'pablo hernandez': 'pablo hernandez',
    'pedro': 'pedro',
    'pedro neto': 'pedro neto',
    'pepe reina': 'pepe reina',
    'richarlison': 'richarlison',
    'roberto': 'roberto',
    'rodri': 'rodri',
    'ruben dias': 'ruben dias',
    'ruben neves': 'ruben neves',
    'ruben vinagre': 'ruben vinagre',
    'rui patricio': 'rui patricio',
    'thiago': 'thiago',
    'thiago alcantara': 'thiago alcantara',
    'thiago silva': 'thiago silva',
    'victor bernth kristiansen': 'victor bernth kristiansen',
    'vinicius souza': 'vinicius souza',
    'willian': 'willian',
    'willian jose': 'willian jose',
    'evanilson': 'evanilson',
    'lyanco': 'lyanco',
    'marc roca': 'marc roca',
    'fabio vieira': 'fabio vieira',
    'hamed junior traore': 'hamed junior traore',
    'jonny castro': 'jonny castro',
    'bruno guimaraes': 'bruno guimaraes',
    'bruno fernandes': 'bruno fernandes',
    'antony': 'antony',
    'bernardo silva': 'bernardo silva',
    'bernardo': 'bernardo',
    'ederson': 'ederson',
    'nelson semedo': 'nelson semedo',
    'diogo dalot': 'diogo dalot',
    'adrian': 'adrian',
    'andrey santos': 'andrey santos',
    'armel bella kotchap': 'armel bella kotchap',
    'benson manuel': 'benson manuel',
    'brandon aguilera': 'brandon aguilera',
    'bryan gil': 'bryan gil',
    'chadi riad': 'chadi riad',
    'cristiano ronaldo': 'cristiano ronaldo',
    'danilo': 'danilo',
    'darwin nunez': 'darwin nunez',
    'deivid washington': 'deivid washington',
    'diego carlos': 'diego carlos',
    'douglas luiz': 'douglas luiz',
    'edson alvarez': 'edson alvarez',
    'ezri konsa': 'ezri konsa',
    'gedson fernandes': 'gedson fernandes',
    'gylfi sigurðsson': 'gylfi sigurðsson',
    'jaden philogene bidace': 'jaden philogene bidace',
    'jesurun rak sakyi': 'jesurun rak sakyi',
    'jesus vallejo': 'jesus vallejo',
    'joachim kayi sanda': 'joachim kayi sanda',
    'johann berg guðmundsson': 'johann berg guðmundsson',
    'julian araujo': 'julian araujo',
    'junior firpo': 'junior firpo',
    'kayky chagas': 'kayky chagas',
    'kayne ramsey': 'kayne ramsey',
    'luis guilherme': 'luis guilherme',
    'mateo joseph': 'mateo joseph',
    'matheus cunha': 'matheus cunha',
    'mathias jørgensen': 'mathias jørgensen',
    'oghenekaro etebo': 'oghenekaro etebo',
    'oriol romeu': 'oriol romeu',
    'pierre højbjerg': 'pierre højbjerg',
    'renan lodi': 'renan lodi',
    'renato veiga': 'renato veiga',
    'ricardo pereira': 'ricardo pereira',
    'samir santos': 'samir santos',
    'savio': 'savio',
    'stefan ortega': 'stefan ortega',
    'youssef chermiti': 'youssef chermiti',
    'yukinari sugawara': 'yukinari sugawara',
    'vitor reis': 'vitor reis',
    'welington': 'welington',
    'gustavo scarpa': 'gustavo scarpa',
    'dan nlundulu': 'dan nlundulu',
    'josh sargent': 'josh sargent',
    'jota': 'jota',
    'abdukodir khusanov': 'abdukodir khusanov',
    'benicio boaitey': 'benicio boaitey',
    'borja baston': 'borja baston',
    'claudio echeverri': 'claudio echeverri',
    'donyell malen': 'donyell malen',
    'eiran cashin': 'eiran cashin',
    'idrissa gana gueye': 'idrissa gana gueye',
    'andres garcia': 'andres garcia',
    'ben gannon-doak': 'ben gannon-doak',
    
    # ============================================
    # ADDITIONAL MAIN->DEFENSIVE MAPPINGS (FINAL 100% PUSH)
    # ============================================
    # These map main dataset cleaned names to defensive cleaned names
    'abdul fatawu': 'abdul fatawu issahaku',
    'ahmed el-sayed hegazi': 'ahmed hegazi',
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'alex moreno lopera': 'alex moreno',
    'ali al-hamadi': 'ali al hamadi',
    'alisson becker': 'alisson',
    'andre tavares gomes': 'andre gomes',
    'anssumane fati vieira': 'ansu fati',
    'armel bella-kotchap': 'armel bella kotchap',
    'benicio baker-boaitey': 'benicio boaitey',
    'bruno borges fernandes': 'bruno fernandes',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'bruno cavaco jordao': 'bruno jordao',
    'bruno andre cavaco jordao': 'bruno jordao',
    'carlos vinicius alves morais': 'carlos vinicius',
    'chadi riad dnanou': 'chadi riad',
    "daniel n'lundulu": 'dan nlundulu',
    'daniel ceballos fernandez': 'dani ceballos',
    'daniel castelo podence': 'daniel podence',
    'daniel drinkwater': 'danny drinkwater',
    'darwin nunez ribeiro': 'darwin nunez',
    'david raya martin': 'david raya',
    'deivid washington de souza eugenio': 'deivid washington',
    'diego carlos santos silva': 'diego carlos',
    'diogo dalot teixeira': 'diogo dalot',
    'douglas luiz soares de paulo': 'douglas luiz',
    'paris maghoma': 'edmond-paris maghoma',
    'edson alvarez velazquez': 'edson alvarez',
    'emerson palmieri dos santos': 'emerson palmieri',
    'emiliano buendia': 'emi buendia',
    'emiliano buendia stati': 'emi buendia',
    'ezri konsa ngoyo': 'ezri konsa',
    'fabio ferreira vieira': 'fabio vieira',
    'gabriel martinelli silva': 'gabriel martinelli',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'gabriel fernando de jesus': 'gabriel jesus',
    'gedson carvalho fernandes': 'gedson fernandes',
    'georges-kevin nkoudou': "georges-kevin n'koudou",
    'giovanni reyna': 'gio reyna',
    'goncalo manuel ganchinho guedes': 'goncalo guedes',
    'gustavo henrique furtado scarpa': 'gustavo scarpa',
    'gylfi sigurdsson': 'gylfi sigurðsson',
    'hamed traore': 'hamed junior traore',
    'ian poveda-ocampo': 'ian poveda',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jaden philogene-bidace': 'jaden philogene bidace',
    'jaden philogene': 'jaden philogene bidace',
    'jhon duran': 'jader duran',  # Colombian spelling
    'jacob sørensen': 'jakob sørensen',
    'javier hernandez balcazar': 'javier hernandez',
    'jesurun rak-sakyi': 'jesurun rak sakyi',
    'jesus vallejo lazaro': 'jesus vallejo',
    'joachim kayi-sanda': 'joachim kayi sanda',
    'joao felix sequeira': 'joao felix',
    'joao palhinha goncalves': 'joao palhinha',
    'joao filipe iria santos moutinho': 'joao moutinho',
    'johann berg gudmundsson': 'johann berg guðmundsson',
    'jonathan castro otto': 'jonny castro',
    'jorge cuenca barreno': 'jorge cuenca',
    'joshua sargent': 'josh sargent',
    'josh acheampong': 'joshua acheampong',
    'juan larios lopez': 'juan larios',
    'julian araujo zuniga': 'julian araujo',
    'junior firpo adames': 'junior firpo',
    'hector junior firpo adames': 'junior firpo',
    'kamari doyle': 'kami doyle',
    'kayky da silva chagas': 'kayky chagas',
    'kayne ramsay': 'kayne ramsey',
    'sung-yueng ki': 'ki sung-yueng',
    'konstantinos tsimikas': 'kostas tsimikas',
    'lucas tolentino coelho de lima': 'lucas paqueta',
    'luis guilherme lira dos santos': 'luis guilherme',
    'lyanco silveira neves vojnovic': 'lyanco',
    'lyanco evangelista silveira neves vojnovic': 'lyanco',
    'mads roerslev rasmussen': 'mads roerslev',
    'marc guiu paz': 'marc guiu',
    'marc roca junque': 'marc roca',
    'marcus myers-harness': 'marcus harness',
    'mateo joseph fernandez': 'mateo joseph',
    'mateo joseph fernandez-regatillo': 'mateo joseph',
    'matheus santos carneiro da cunha': 'matheus cunha',
    'matheus franca de oliveira': 'matheus franca',
    'mathias jorgensen': 'mathias jørgensen',
    'murillo santiago costa dos santos': 'murillo',
    'murillo costa dos santos': 'murillo',
    'nelson cabral semedo': 'nelson semedo',
    "nico o'reilly": "nico o'reilly",
    'nuno varela tavares': 'nuno tavares',
    'oghenekaro peter etebo': 'oghenekaro etebo',
    'oriol romeu vidal': 'oriol romeu',
    'pedro lomba neto': 'pedro neto',
    'pedro cardoso de lima': 'pedro lima',
    'pierre-emile højbjerg': 'pierre højbjerg',
    'przemyslaw placheta': 'przemysław płacheta',
    'renato palma veiga': 'renato veiga',
    'ricardo barbosa pereira': 'ricardo pereira',
    'richarlison de andrade': 'richarlison',
    'rodrigo moreno': 'rodrigo',
    'rodrigo martins gomes': 'rodrigo gomes',
    'rodrigo muniz carvalho': 'rodrigo muniz',
    'rodrigo duarte ribeiro': 'rodrigo ribeiro',
    'ruben gato alves dias': 'ruben dias',
    'ruben santos gato alves dias': 'ruben dias',
    'ruben da silva neves': 'ruben neves',
    'ruben diogo da silva neves': 'ruben neves',
    'ruben nascimento vinagre': 'ruben vinagre',
    'ruben goncalo silva nascimento vinagre': 'ruben vinagre',
    'sam szmodics': 'sammie szmodics',
    'stefan ortega moreno': 'stefan ortega',
    'tariqe fosu-henry': 'tariqe fosu',
    'thiago thiago': 'thiago',
    'thiago alcantara do nascimento': 'thiago alcantara',
    'thiago emiliano da silva': 'thiago silva',
    'victor kristiansen': 'victor bernth kristiansen',
    'vini de souza costa': 'vinicius souza',
    'vitalii mykolenko': 'vitaliy mykolenko',
    'william fish': 'will fish',
    'willian jose da silva': 'willian jose',
    'willian borges da silva': 'willian',
    'youssef ramalho chermiti': 'youssef chermiti',
    'gustavo nunes fernandes gomes': 'gustavo nunes',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'carlos henrique casimiro': 'casemiro',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'fernando luiz rosa': 'fernandinho',
    'rafael dias belloli': 'raphinha',
    'raphael dias belloli': 'raphinha',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'kim ji-soo': 'kim jisoo',
    'sugawara yukinari': 'yukinari sugawara',
    'vitor de oliveira nunes dos reis': 'vitor reis',
    'welington damascena santos': 'welington',
    'savio moreira de oliveira': 'savio',
    "savio 'savinho' moreira de oliveira": 'savio',
    'francisco evanilson de lima barbosa': 'evanilson',
    'jose reina': 'pepe reina',
    'jose manuel reina': 'pepe reina',
    'rui pedro dos santos patricio': 'rui patricio',
    'jose ignacio peleteiro romallo': 'jota',
    'jorge luiz frello filho': 'jorginho',
    'thakgalo leshabela': 'khanya leshabela',
    'samir caetano de souza santos': 'samir santos',
    # 'kenny tete': 'tete',  # REMOVED - defensive has 'kenny tete'
    'robert kenedy nunes do nascimento': 'kenedy',
    'norberto bercique gomes betuncal': 'beto',
    'andre trindade da costa neto': 'andre',
    'norberto murara neto': 'neto',
    'igor thiago nascimento rodrigues': 'igor',
    'igor julio dos santos de paulo': 'igor',
    'igor jesus maciel da cruz': 'igor',
    'ederson santana de moraes': 'ederson',
    'emerson aparecido leite de souza junior': 'emerson',
    'emerson leite de souza junior': 'emerson',
    'felipe augusto de almeida monteiro': 'felipe',
    'felipe rodrigues da silva': 'felipe anderson',
    'danilo dos santos de oliveira': 'danilo',
    'danilo luiz da silva': 'danilo',
    'bernard anicio caldeira duarte': 'bernard',
    'allan marques loureiro': 'allan',
    'adrian san miguel del castillo': 'adrian',
    'diego da silva costa': 'diego costa',
    'cristiano ronaldo dos santos aveiro': 'cristiano ronaldo',
    'lucas moura': 'lucas moura',
    # 'lucas torreira': 'lucas moura',  # REMOVED - they are different players
    'joao pedro junqueira de jesus': 'joao pedro',
    'joao pedro ferreira silva': 'joao pedro',
    'joao pedro ferreira da silva': 'joao pedro',
    'joelinton cassio apolinario de lira': 'joelinton',
    'diogo jota': 'diogo jota',  # Keep Diogo Jota as-is
    'david luiz moreira marinho': 'david luiz',
    'rodrigo hernandez': 'rodri',
    'rodrigo hernandez cascante': 'rodri',
    'marquinhos': 'marquinhos',  # Self-map (no FPL data)
    'chiquinho': 'chiquinho',  # Self-map (no FPL data)
    'vitinha': 'vitinha',  # Self-map
    'vitinho': 'vitinho',  # Self-map
    'trezeguet': 'trezeguet',  # Self-map
    'cafu': 'cafu',  # Self-map
    'cucho': 'cucho',  # Self-map
    'angelino': 'angelino',  # Self-map
    'beto': 'beto',  # Self-map
    'morato': 'morato',  # Self-map
    
    # ============================================
    # FINAL PUSH - REMAINING 43 MISSING NAMES
    # ============================================
    # These are names in defensive data that clean correctly but
    # need exact matches from main data
    
    # Players where main has shortened name, defensive has full name
    'idrissa gueye': 'idrissa gana gueye',  # Main has short name
    'borja baston': 'borja baston',  # accent difference
    'albert gronbaek': 'albert grønbaek',  # accent difference
    'nicolas gonzalez iglesias': 'nicolas gonzalez',  # Different player?
    
    # Ensure these identical names pass through correctly (self-mappings)
    'abdukodir khusanov': 'abdukodir khusanov',
    'alex palmer': 'alex palmer',
    'andres garcia': 'andres garcia',
    'donyell malen': 'donyell malen',
    'claudio echeverri': 'claudio echeverri',
    'eiran cashin': 'eiran cashin',
    'kenny tete': 'kenny tete',
    'rodrigo bentancur': 'rodrigo bentancur',
    'marshall munetsi': 'marshall munetsi',
    'omar marmoush': 'omar marmoush',
    'mathys tel': 'mathys tel',
    'michael kayode': 'michael kayode',
    'patrick dorgu': 'patrick dorgu',
    'romain esse': 'romain esse',
    'tyler fredricson': 'tyler fredricson',
    'zain silcott-duberry': 'zain silcott-duberry',
    'lucas torreira': 'lucas torreira',
    'pedro rodriguez ledesma': 'pedro',
    'jota silva': 'jota silva',
    'marco asensio': 'marco asensio',
    'mateus mane': 'mateus mane',
    
    # Players not in FPL (defensive only) - self-map to preserve
    'chidozie obi-martin': 'chidozie obi-martin',
    'chiquinho': 'chiquinho',
    'mathis amougou': 'mathis amougou',
    'nasser djiga': 'nasser djiga',
    'olabade aluko': 'olabade aluko',
    'shumaira mheuka': 'shumaira mheuka',
    'trezeguet': 'trezeguet',
    'vitinho': 'vitinho',
    'vitor reis': 'vitor reis',
    'woyo coulibaly': 'woyo coulibaly',
    'harry howell': 'harry howell',
    'jake evans': 'jake evans',
    'jay robinson': 'jay robinson',
    'jeremy monga': 'jeremy monga',
    'ben gannon-doak': 'ben gannon-doak',
    'roberto': 'roberto',
    'pedro': 'pedro',
}

# Apply mappings to MAIN dataset using map() for memory efficiency
df_main['join_name'] = df_main['join_name'].map(lambda x: manual_nickname_map.get(x, x))
print(f"Applied {len(manual_nickname_map)} manual nickname mappings")

print("\n" + "="*80)
print("STEP 3: AUTOMATED SMART MATCHING")
print("="*80)

# Get unique names from both
valid_def_names = set(df_def['join_name'].unique())
main_unique = df_main['join_name'].unique()
missing_names = [n for n in main_unique if n not in valid_def_names]

print(f"Names in main not yet matching defensive: {len(missing_names)}")

name_mapping = {}

# LOGIC A: Substring matching (Long name contains short name)
print("\nApplying substring matching...")
for m_name in missing_names:
    m_tokens = set(m_name.split())
    candidates = []
    
    for d_name in valid_def_names:
        d_tokens = set(d_name.split())
        # Check if defensive name tokens are subset of main name tokens
        if d_tokens.issubset(m_tokens) and len(d_tokens) >= 2:
            candidates.append(d_name)
    
    if candidates:
        # Pick the longest match (most specific)
        best_match = max(candidates, key=len)
        name_mapping[m_name] = best_match

print(f"  Found {len(name_mapping)} matches via substring")

# LOGIC B: Fuzzy matching for remaining
remaining_missing = [n for n in missing_names if n not in name_mapping]
def_name_list = list(valid_def_names)

print("Applying fuzzy matching...")
for m_name in remaining_missing:
    matches = difflib.get_close_matches(m_name, def_name_list, n=1, cutoff=0.85)
    if matches:
        name_mapping[m_name] = matches[0]

print(f"  Found {len([n for n in name_mapping if n in remaining_missing])} matches via fuzzy")

# Apply the automatic mappings using map() for memory efficiency
df_main['join_name'] = df_main['join_name'].map(lambda x: name_mapping.get(x, x))
print(f"\nTotal automatic matches applied: {len(name_mapping)}")

print("\n" + "="*80)
print("STEP 4: COVERAGE REPORT")
print("="*80)

# Final coverage check
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"\nUnique Defensive Names:     {total_def_names}")
print(f"Found in Main DataFrame:    {matched_count}")
print(f"Defensive Name Coverage:    {coverage_pct:.2f}%")
print(f"Missing from Main:          {len(missing_def_names)}")

if len(missing_def_names) > 0 and len(missing_def_names) <= 30:
    print(f"\nStill missing ({len(missing_def_names)} names):")
    for name in sorted(missing_def_names):
        orig = df_def[df_def['join_name'] == name]['name'].iloc[0]
        seasons = df_def[df_def['join_name'] == name]['season'].unique()
        print(f"  '{name}' <- '{orig}' ({list(seasons)})")
elif len(missing_def_names) > 30:
    print(f"\n(Showing ALL {len(missing_def_names)} missing names)")
    for name in sorted(missing_def_names):
        orig = df_def[df_def['join_name'] == name]['name'].iloc[0]
        print(f"  '{name}' <- '{orig}'")

### coverage report:

In [ ]:
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")

# now the real merging:

In [ ]:
# ==========================================
# 2. THE MERGE & SMART VALIDATION
# ==========================================
# IMPORTANT: We now use game_number instead of GW for merging
# This handles postponed matches correctly by matching on chronological game order
print("--- MERGE DIAGNOSTICS (Using game_number) ---")

# 0. CREATE game_number FOR DEFENSIVE DATA
# Convert game_date to datetime and sort to create chronological game_number
if 'game_number' not in df_def.columns:
    print("Creating game_number for defensive data...")
    df_def['game_date'] = pd.to_datetime(df_def['game_date'], errors='coerce')
    
    # Sort by player, season, and game_date, then assign game number
    df_def = df_def.sort_values(['join_name', 'join_season', 'game_date'])
    df_def['game_number'] = df_def.groupby(['join_name', 'join_season']).cumcount() + 1
    
    print(f"  Created game_number (1-{df_def['game_number'].max()}) for defensive data")

# 1. PREPARE DEFENSIVE DATA
cols_cbi = ['clearances', 'blocks', 'interceptions']
df_def[cols_cbi] = df_def[cols_cbi].fillna(0)

if 'clearances_blocks_interceptions' not in df_def.columns:
    df_def['clearances_blocks_interceptions'] = (
        df_def['clearances'] + df_def['blocks'] + df_def['interceptions']
    )

# Select Merge Subset - NOW USING game_number INSTEAD OF GW
def_subset = df_def[[
    'join_name', 'game_number', 'join_season', 
    'tackles', 'clearances_blocks_interceptions'
]].rename(columns={
    'tackles': 'tackles_new', 
    'clearances_blocks_interceptions': 'cbi_new'
})

# ---------------------------------------------------------
# SMART METRIC: "Can we match it?"
# ---------------------------------------------------------
# Using game_number for matching ensures correct alignment even with postponed matches
main_keys = set(zip(df_main['join_name'], df_main['game_number'], df_main['join_season']))
def_keys = set(zip(def_subset['join_name'], def_subset['game_number'], def_subset['join_season']))

# The Intersection: These are the rows that SHOULD merge successfully
possible_matches = main_keys.intersection(def_keys)
print(f"Total Rows in Main: {len(df_main)}")
print(f"Rows with available Defensive Data: {len(possible_matches)}")

# 2. PERFORM LEFT MERGE - NOW ON game_number
merged_df = pd.merge(
    df_main, 
    def_subset, 
    on=['join_name', 'game_number', 'join_season'], 
    how='left'
)

# ---------------------------------------------------------
# REAL VALIDATION: DID THE MERGE WORK?
# ---------------------------------------------------------
merged_df['key_tuple'] = list(zip(merged_df['join_name'], merged_df['game_number'], merged_df['join_season']))
should_have_data = merged_df[merged_df['key_tuple'].isin(possible_matches)]

# Check if they are actually filled
successful_merges = should_have_data['tackles_new'].notna().sum()
technical_success_rate = (successful_merges / len(should_have_data)) * 100 if len(should_have_data) > 0 else 0

print(f"\nTechnical Merge Success Rate: {technical_success_rate:.2f}%")
print("(This should be 100%. It means every row that existed in the source was successfully merged.)")

# ==========================================
# 3. UPDATE STATS & FINAL REPORT
# ==========================================
# Update Tackles
merged_df['tackles'] = np.where(
    merged_df['tackles_new'].notna(), 
    merged_df['tackles_new'], 
    np.where(merged_df['minutes'] == 0, 0, merged_df['tackles'].fillna(0))
)

# Update CBI
old_cbi = merged_df['clearances_blocks_interceptions'] if 'clearances_blocks_interceptions' in merged_df.columns else 0
merged_df['clearances_blocks_interceptions'] = np.where(
    merged_df['cbi_new'].notna(), 
    merged_df['cbi_new'], 
    np.where(merged_df['minutes'] == 0, 0, old_cbi)
)

# Clean up temps - keep game_number as it's useful for feature engineering
merged_df.drop(columns=['tackles_new', 'cbi_new', 'join_name', 'join_season', 'is_matched', 'key_tuple'], inplace=True, errors='ignore')

print("\n✓ Merge complete using game_number for correct chronological alignment")

# here validate that after merging we made the results in all seasons data

In [ ]:
# ==========================================
# STEP 4: COMBINE MERGED DATA BACK INTO FULL DATASET
# ==========================================
# Problem: We filtered df_main to only seasons with defensive data (2019-20 to 2024-25)
# Solution: Reload the full dataset and combine:
#   - Seasons 2016-17 to 2018-19: Already have real defensive stats
#   - Seasons 2019-20 to 2024-25: Use merged_df with newly merged defensive stats
#   - Season 2025-26: Keep as-is (if not in defensive data)

print("="*80)
print("STEP 4: Combining merged data back into full dataset")
print("="*80)

# 1. Reload the full all_seasons_data (before we filtered it)
df_full = pd.read_csv("all_seasons_data.csv")
print(f"Full dataset shape: {df_full.shape}")
print(f"Seasons in full dataset: {sorted(df_full['season'].unique())}")

# 2. Get the seasons that were merged with defensive data
merged_seasons = merged_df['season'].unique().tolist()
print(f"\nSeasons that were merged with defensive stats: {merged_seasons}")

# 3. Get the seasons that were NOT merged (already have defensive data or no data available)
seasons_not_merged = [s for s in df_full['season'].unique() if s not in merged_seasons]
print(f"Seasons NOT merged (already have defensive data): {seasons_not_merged}")

# 4. Extract the non-merged seasons from the full dataset
df_not_merged = df_full[df_full['season'].isin(seasons_not_merged)].copy()
print(f"Records from non-merged seasons: {len(df_not_merged):,}")

# 5. Ensure both DataFrames have the same columns
# Add game_number to non-merged seasons if missing (set to None for older seasons)
if 'game_number' not in df_not_merged.columns:
    df_not_merged['game_number'] = None
    
# Align columns - get the union of both column sets
all_columns = list(set(merged_df.columns) | set(df_not_merged.columns))
for col in all_columns:
    if col not in merged_df.columns:
        merged_df[col] = None
    if col not in df_not_merged.columns:
        df_not_merged[col] = None

# 6. Combine merged seasons with non-merged seasons
all_seasons_final = pd.concat([df_not_merged, merged_df], ignore_index=True)

# 7. Sort by season and element for consistency
all_seasons_final = all_seasons_final.sort_values(['season', 'element', 'GW']).reset_index(drop=True)

print(f"\n✅ Final combined dataset shape: {all_seasons_final.shape}")
print(f"Seasons in final dataset: {sorted(all_seasons_final['season'].unique())}")

# 8. Verify record counts by season
print("\nRecords per season:")
print(all_seasons_final.groupby('season').size().sort_index())

## Step 5: Recalculate defensive_contribution and modify points for merged seasons

In [ ]:
# ==========================================
# STEP 5: RECALCULATE DEFENSIVE CONTRIBUTION & MODIFY POINTS
# ==========================================
# Now that we have the real defensive stats for 2019-20 to 2024-25,
# we need to recalculate defensive_contribution and apply the point modifications

print("="*80)
print("STEP 5: Recalculating defensive contribution for merged seasons")
print("="*80)

# 1. Define the defensive contribution calculation function
def calculate_defensive_contribution_row(row):
    """Calculate defensive contribution based on position"""
    position = row['position']
    cbi = row.get('clearances_blocks_interceptions', 0) or 0
    tackles = row.get('tackles', 0) or 0
    recoveries = row.get('recoveries', 0) or 0
    
    if position == 'DEF':
        return cbi + tackles
    elif position in ['MID', 'FWD']:
        return cbi + tackles + recoveries
    else:  # GK or unknown
        return 0

# 2. Recalculate defensive_contribution for seasons that were merged
# (2019-20 to 2024-25 now have real defensive stats)
merged_season_mask = all_seasons_final['season'].isin(merged_seasons)

print(f"Recalculating defensive_contribution for {merged_season_mask.sum():,} records...")

all_seasons_final.loc[merged_season_mask, 'defensive_contribution'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_defensive_contribution_row, axis=1)

# 3. Apply point modification to merged seasons (as per FPL rules)
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12

def calculate_modified_points(row):
    """Add 2 bonus points for defensive contribution threshold"""
    points = row['total_points']
    position = row['position']
    def_contrib = row.get('defensive_contribution', 0) or 0
    
    if position == 'DEF' and def_contrib >= 10:
        points += 2
    elif position in ['MID', 'FWD'] and def_contrib >= 12:
        points += 2
    return points

# Only modify points for the merged seasons (2019-20 to 2024-25)
# Seasons 2016-17 to 2018-19 already had points modified earlier
print("Applying point modifications for defensive contributions...")

all_seasons_final.loc[merged_season_mask, 'total_points'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_modified_points, axis=1)

print("✅ Defensive contribution recalculated and points modified")

# 4. Verify the results
print("\nSample of defensive stats after recalculation:")
sample_cols = ['name', 'season', 'position', 'tackles', 'clearances_blocks_interceptions', 
               'defensive_contribution', 'total_points']
available_cols = [c for c in sample_cols if c in all_seasons_final.columns]
print(all_seasons_final[all_seasons_final['season'] == '2023-24'][available_cols].sample(10))

## Step 6: Save the final complete dataset

In [ ]:
# ==========================================
# STEP 6: SAVE THE FINAL COMPLETE DATASET
# ==========================================
print("="*80)
print("STEP 6: Saving final dataset")
print("="*80)

# Save the complete dataset with all seasons and merged defensive stats
output_path = 'all_seasons_data_final.csv'
all_seasons_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Saved: {output_path}")
print(f"   Total records: {len(all_seasons_final):,}")
print(f"   Total columns: {len(all_seasons_final.columns)}")
print(f"   Seasons: {sorted(all_seasons_final['season'].unique())}")

# Summary statistics
print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"\nRecords by season:")
print(all_seasons_final.groupby('season').size().sort_index())

print(f"\nDefensive stats coverage (non-zero defensive_contribution):")
def_contrib_stats = all_seasons_final.groupby('season').apply(
    lambda x: (x['defensive_contribution'] > 0).sum() / len(x) * 100
)
print(def_contrib_stats.round(1).to_string())

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE - READY FOR FEATURE ENGINEERING")
print("="*80)

# now we make the previous game and that stuff:

### load the dataset

In [ ]:
all_seasons_data = pd.read_csv('all_seasons_data_final.csv')

## defining the function to add the rollback stats

In [ ]:
def add_previous_game_stats(df, n_gameweeks=5, use_cross_season=False):
    """
    Add columns for each stat for the previous N games to each player.
    
    IMPORTANT: Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This handles postponed matches where a GW5 match might be played during GW20.
    
    When use_cross_season=True, uses 'name' instead of 'element' because:
    - 'element' IDs are only unique within a season (different IDs for same player across seasons)
    - 'name' is consistent across seasons, enabling true cross-season data continuity
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing player statistics with columns including 'name', 'element', 'game_number', 'season'
    n_games : int, default=5
        Number of previous games to include
    use_cross_season : bool, default=False
        If True, carry over stats from previous season (uses 'name' for grouping)
        If False, stats only come from within the same season (Game 1 will have NaN for previous stats)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with additional columns for previous game statistics
    """
    # Columns to exclude from previous gameweek calculation
    exclude_columns = ['name', 'element', 'GW', 'game_number', 'position', 'team', 
                       'season', 'fixture', 'kickoff_time', 'round', ]
    
    # Get stat columns (only numeric columns except the excluded ones)
    stat_columns = [col for col in df.columns 
                    if col not in exclude_columns and pd.api.types.is_numeric_dtype(df[col])]
    
    # Determine groupby columns and sort order based on cross_season parameter
    if use_cross_season:
        # Use 'name' for cross-season continuity (element IDs change between seasons)
        # Sort by name, season, game_number for correct chronological order
        df_sorted = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['name']
    else:
        # Group by element and season - stats only within same season
        df_sorted = df.sort_values(['element', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['element', 'season']
    
    # Create a copy to avoid modifying the original
    df_result = df_sorted.copy()
    
    # For each stat column, create N previous game columns
    for stat in stat_columns:
        for i in range(1, n_gameweeks + 1):
            col_name = f'{stat}_prev_{i}'
            # Shift by i positions for each player (based on game_number order)
            df_result[col_name] = df_result.groupby(group_cols, sort=False)[stat].shift(i)
    
    return df_result

### apllying the function

In [ ]:
# Load data if needed
# all_seasons_data = pd.read_csv('all_seasons_data.csv', index_col=0)

# Apply the function to create lagged features
# Parameters:
#   n_gameweeks: Number of previous gameweeks to include (default=5)
#   use_cross_season: Whether to carry stats across seasons (default=False)
all_seasons_data_with_prev = add_previous_game_stats(all_seasons_data, n_gameweeks=5, use_cross_season=True)

# Check the result
print("Original shape:", all_seasons_data.shape)
print("New shape:", all_seasons_data_with_prev.shape)
print("\nSample of new columns:")
print(all_seasons_data_with_prev.filter(regex='_prev_').columns.tolist()[:20])

## Feature Engineering: Opponent Strength
Calculate opponent strength metrics based on team performance to capture fixture difficulty.

In [ ]:
def add_opponent_strength_features(df, rolling_windows=[3, 5]):
    """
    Add opponent strength features and team rolling stats based on historical performance.
    
    IMPORTANT: Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This handles postponed matches where teams don't play in certain gameweeks.
    
    Features added:
    - Team rolling features: goals_scored, goals_conceded, clean_sheets, total_points (for each window)
    - Opponent rolling features: same metrics for opponent team
    - Strength features: defensive_strength, offensive_strength, overall_team_strength
    - Advantage features: offensive_advantage, defensive_advantage, overall_advantage
    - Difficulty: opponent_difficulty rating
    
    NaN Behavior:
    - Uses min_periods=1, so rolling computes with available data (even if < window size)
    - shift(1) causes Game 1 of each season to have NaN (no prior game exists)
    - NaN values are NOT filled - they remain as NaN for transparency
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data with team and opponent_team columns
    rolling_windows : list, default=[3, 5]
        List of rolling window sizes for team stats
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added team/opponent strength and rolling features
    """
    
    # Sort in-place to save memory, then reset index
    df.sort_values(['season', 'game_number', 'team'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    # Step 1: Calculate team-level aggregates per game_number
    # We use game_number to ensure correct ordering even with postponed matches
    # goals_scored uses 'sum' because we sum all player goals
    # goals_conceded uses 'max' because it's team-level (same for all players)
    # clean_sheets is derived from goals_conceded (not from player data, as players 
    # can have CS=1 if subbed after 60min but before team conceded)
    team_stats = df.groupby(['season', 'game_number', 'team']).agg({
        'goals_scored': 'sum',           # Sum of all player goals = team goals
        'goals_conceded': 'max',         # Team-level stat (same for all players)
        'total_points': 'sum',           # Sum of all player FPL points
    }).reset_index()
    
    # Rename for clarity
    team_stats.columns = ['season', 'game_number', 'team', 'team_goals_scored', 
                          'team_goals_conceded', 'team_total_points']
    
    # Derive team clean sheet from goals_conceded (CS=1 only if goals_conceded=0)
    team_stats['team_clean_sheet'] = (team_stats['team_goals_conceded'] == 0).astype(int)
    
    # Step 2: Sort by team, season, and game_number for correct rolling calculation
    team_stats.sort_values(['team', 'season', 'game_number'], inplace=True)
    
    # Step 3: Calculate rolling averages for each window size
    # min_periods=1 means rolling calculates even with fewer values than window size:
    #   - Game 2's rolling_3 uses only Game 1 (1 value, not 3)
    #   - Game 3's rolling_3 uses Games 1-2 (2 values, not 3)
    #   - Game 4's rolling_3 uses Games 1-3 (full 3 values)
    # shift(1) pushes values forward, so we use PAST games only (Game 1 → NaN)
    team_rolling_cols = ['team_goals_scored', 'team_goals_conceded', 'team_clean_sheet', 'team_total_points']
    
    for window in rolling_windows:
        for col in team_rolling_cols:
            team_stats[f'{col}_rolling_{window}'] = (
                team_stats.groupby(['team', 'season'])[col]
                .transform(lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))
            )
    
    # Step 4: Calculate strength ratings (using rolling_3 as primary window for strength)
    primary_window = rolling_windows[0]  # Use first window (3) for strength calculations
    
    # Defensive strength = inverse of goals conceded (lower conceded = higher strength)
    team_stats['defensive_strength'] = (
        1 / (team_stats[f'team_goals_conceded_rolling_{primary_window}'] + 1)
    )
    
    # Offensive strength = goals scored rolling average
    team_stats['offensive_strength'] = (
        team_stats[f'team_goals_scored_rolling_{primary_window}']
    )
    
    # Overall team strength = total points rolling average
    team_stats['overall_team_strength'] = (
        team_stats[f'team_total_points_rolling_{primary_window}']
    )
    
    # Step 5: Prepare columns for merge (NO NaN filling - keep NaN for transparency)
    own_team_cols = ['season', 'game_number', 'team', 
                     'defensive_strength', 'offensive_strength', 'overall_team_strength']
    for window in rolling_windows:
        for col in team_rolling_cols:
            own_team_cols.append(f'{col}_rolling_{window}')
    
    # Step 6: Create opponent stats dataframe with 'opponent_' prefix
    opponent_stats = team_stats[['season', 'game_number', 'team', 
                                  'defensive_strength', 'offensive_strength', 'overall_team_strength'] + 
                                 [f'{col}_rolling_{w}' for w in rolling_windows for col in team_rolling_cols]].copy()
    
    rename_dict = {'team': 'opponent_team'}
    for col in opponent_stats.columns:
        if col not in ['season', 'game_number', 'team']:
            rename_dict[col] = 'opponent_' + col
    opponent_stats.rename(columns=rename_dict, inplace=True)
    
    # Step 7: Merge own team stats
    df = df.merge(
        team_stats[own_team_cols],
        on=['season', 'game_number', 'team'],
        how='left'
    )
    
    # Step 8: Merge opponent stats
    df = df.merge(
        opponent_stats,
        on=['season', 'game_number', 'opponent_team'],
        how='left'
    )
    
    # Step 9: Calculate relative strength features (advantage)
    # These will be NaN if either team's strength is NaN
    df['offensive_advantage'] = df['offensive_strength'] - df['opponent_defensive_strength']
    df['defensive_advantage'] = df['defensive_strength'] - df['opponent_offensive_strength']
    df['overall_advantage'] = df['overall_team_strength'] - df['opponent_overall_team_strength']
    
    # Step 10: Add difficulty rating (normalized)
    # Higher value = more difficult opponent
    df['opponent_difficulty'] = (
        df['opponent_overall_team_strength'] / df['overall_team_strength']
    )
    
    # Clean up memory
    del team_stats, opponent_stats
    
    return df

### Apply Opponent Strength Features
Add team and opponent strength metrics to the dataset.

In [ ]:
# Free up memory before applying opponent strength features
import gc

# Delete the previous version to free memory
if 'all_seasons_data_with_opponent' in dir():
    del all_seasons_data_with_opponent
gc.collect()

# Apply opponent strength features to a copy of the data (to avoid modifying original)
all_seasons_data_with_opponent = add_opponent_strength_features(
    all_seasons_data_with_prev.copy(), 
    rolling_windows=[3, 5]  # Rolling windows for team stats
)

# Check new features
print("Shape after adding opponent features:", all_seasons_data_with_opponent.shape)
print("\nNew team/opponent columns added:")
team_opponent_cols = [col for col in all_seasons_data_with_opponent.columns 
                      if any(x in col.lower() for x in ['team_goals', 'team_clean', 'team_total_points', 
                                                         'opponent', 'advantage', 'difficulty', 'strength'])]
for col in sorted(team_opponent_cols):
    print(f"  - {col}")

In [ ]:
# ============================================================================
# VALIDATION: Team Rolling Features & NaN Analysis
# ============================================================================
print("=" * 80)
print("VALIDATION: Team Rolling Features for Opponent Strength")
print("=" * 80)

# 1. Check a specific team's rolling stats across first 6 games
team_to_check = "Arsenal"
season_to_check = "2023-24"

arsenal_games = all_seasons_data_with_opponent[
    (all_seasons_data_with_opponent['team'] == team_to_check) & 
    (all_seasons_data_with_opponent['season'] == season_to_check)
][['game_number', 'team', 'team_goals_scored_rolling_3', 'team_goals_scored_rolling_5',
   'team_goals_conceded_rolling_3', 'team_goals_conceded_rolling_5',
   'team_clean_sheet_rolling_3', 'team_clean_sheet_rolling_5']].drop_duplicates().sort_values('game_number')

print(f"\n1. {team_to_check}'s First 6 Games in {season_to_check} - Team Rolling Stats:")
print("-" * 80)
display(arsenal_games.head(6))

# 2. Show actual game results with DERIVED team clean sheet
print(f"\n2. {team_to_check}'s Actual Game Results (with DERIVED team_clean_sheet):")
print("-" * 80)
arsenal_actual = all_seasons_data_with_opponent[
    (all_seasons_data_with_opponent['team'] == team_to_check) & 
    (all_seasons_data_with_opponent['season'] == season_to_check)
].groupby('game_number').agg({
    'goals_scored': 'sum',
    'goals_conceded': 'max',
}).reset_index().sort_values('game_number')

# Derive team_clean_sheet correctly: 1 only if goals_conceded == 0
arsenal_actual['team_clean_sheet'] = (arsenal_actual['goals_conceded'] == 0).astype(int)
display(arsenal_actual.head(6))

print("\nNote: team_clean_sheet is derived as 1 ONLY when goals_conceded = 0")
print("The raw 'clean_sheets' column from player data can be 1 even when team conceded")
print("(if player was subbed after 60min but before team conceded)")

# 3. Explain NaN behavior with min_periods=1
print("\n3. NaN Behavior Explanation (min_periods=1):")
print("-" * 80)
print("""
With min_periods=1 and shift(1):
- Game 1: NaN (no previous game after shift)
- Game 2: Uses Game 1 only (1 value, not full 3 for rolling_3)
- Game 3: Uses Games 1-2 (2 values, not full 3 for rolling_3)
- Game 4: Uses Games 1-3 (FULL 3 values for rolling_3)

So rolling_3 at Game 2 = Game 1's value
   rolling_3 at Game 3 = avg(Game 1, Game 2)
   rolling_3 at Game 4 = avg(Game 1, Game 2, Game 3) ← First true rolling_3
""")

# 4. Verify the calculation manually
print("4. Manual Verification of Rolling Calculation:")
print("-" * 80)
if len(arsenal_actual) >= 4:
    g1_goals = arsenal_actual[arsenal_actual['game_number'] == 1]['goals_scored'].values[0]
    g2_goals = arsenal_actual[arsenal_actual['game_number'] == 2]['goals_scored'].values[0]
    g3_goals = arsenal_actual[arsenal_actual['game_number'] == 3]['goals_scored'].values[0]
    
    expected_g2 = g1_goals  # Only Game 1
    expected_g3 = (g1_goals + g2_goals) / 2  # Avg of Games 1-2
    expected_g4 = (g1_goals + g2_goals + g3_goals) / 3  # Avg of Games 1-3
    
    actual_g2 = arsenal_games[arsenal_games['game_number'] == 2]['team_goals_scored_rolling_3'].values[0]
    actual_g3 = arsenal_games[arsenal_games['game_number'] == 3]['team_goals_scored_rolling_3'].values[0]
    actual_g4 = arsenal_games[arsenal_games['game_number'] == 4]['team_goals_scored_rolling_3'].values[0]
    
    print(f"Game 2 rolling_3: Expected={expected_g2:.2f}, Actual={actual_g2:.2f} {'✓' if abs(expected_g2-actual_g2)<0.01 else '✗'}")
    print(f"Game 3 rolling_3: Expected={expected_g3:.2f}, Actual={actual_g3:.2f} {'✓' if abs(expected_g3-actual_g3)<0.01 else '✗'}")
    print(f"Game 4 rolling_3: Expected={expected_g4:.2f}, Actual={actual_g4:.2f} {'✓' if abs(expected_g4-actual_g4)<0.01 else '✗'}")

# 5. Verify clean sheet rolling calculation
print("\n5. Clean Sheet Rolling Verification:")
print("-" * 80)
g1_cs = arsenal_actual[arsenal_actual['game_number'] == 1]['team_clean_sheet'].values[0]
g2_cs = arsenal_actual[arsenal_actual['game_number'] == 2]['team_clean_sheet'].values[0]
g3_cs = arsenal_actual[arsenal_actual['game_number'] == 3]['team_clean_sheet'].values[0]

expected_cs_g2 = g1_cs  # Only Game 1
expected_cs_g3 = (g1_cs + g2_cs) / 2  # Avg of Games 1-2
expected_cs_g4 = (g1_cs + g2_cs + g3_cs) / 3  # Avg of Games 1-3

actual_cs_g2 = arsenal_games[arsenal_games['game_number'] == 2]['team_clean_sheet_rolling_3'].values[0]
actual_cs_g3 = arsenal_games[arsenal_games['game_number'] == 3]['team_clean_sheet_rolling_3'].values[0]
actual_cs_g4 = arsenal_games[arsenal_games['game_number'] == 4]['team_clean_sheet_rolling_3'].values[0]

print(f"Game 1: goals_conceded={arsenal_actual[arsenal_actual['game_number']==1]['goals_conceded'].values[0]} → team_clean_sheet={g1_cs}")
print(f"Game 2: goals_conceded={arsenal_actual[arsenal_actual['game_number']==2]['goals_conceded'].values[0]} → team_clean_sheet={g2_cs}")
print(f"Game 3: goals_conceded={arsenal_actual[arsenal_actual['game_number']==3]['goals_conceded'].values[0]} → team_clean_sheet={g3_cs}")
print()
print(f"Game 2 CS rolling_3: Expected={expected_cs_g2:.2f}, Actual={actual_cs_g2:.2f} {'✓' if abs(expected_cs_g2-actual_cs_g2)<0.01 else '✗'}")
print(f"Game 3 CS rolling_3: Expected={expected_cs_g3:.2f}, Actual={actual_cs_g3:.2f} {'✓' if abs(expected_cs_g3-actual_cs_g3)<0.01 else '✗'}")
print(f"Game 4 CS rolling_3: Expected={expected_cs_g4:.2f}, Actual={actual_cs_g4:.2f} {'✓' if abs(expected_cs_g4-actual_cs_g4)<0.01 else '✗'}")

## Feature Engineering: Rolling Averages (Player Form)
Calculate moving averages to capture short-term and long-term player performance trends.

In [ ]:
def add_rolling_player_stats(df, windows=[3, 5, 10]):
    """
    Add rolling averages for player performance metrics to capture form.
    
    IMPORTANT: Uses 'name' instead of 'element' for cross-season continuity.
    The 'element' ID is only unique within a season, but 'name' allows us to
    track player form across season boundaries.
    
    Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This ensures rolling averages are calculated based on actual game sequence,
    not gameweek numbers (which can be out of order due to postponements).
    
    The .shift(1) is applied INSIDE the groupby using transform() to ensure
    the shift only happens within each player's data, preventing data leakage
    between different players.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data
    windows : list, default=[3, 5, 10]
        List of rolling window sizes (3-game, 5-game, 10-game form)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added rolling average features
    """
    
    # Key performance metrics to calculate rolling averages for
    rolling_stats = ['total_points', 'goals_scored', 'assists', 'minutes', 
                     'bonus', 'bps', 'clean_sheets', 'saves', 
                     'ict_index', 'creativity', 'threat', 'influence']
    
    # Sort by player NAME (consistent across seasons), then season and game_number
    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
    
    # Calculate rolling averages for each window size
    for window in windows:
        for stat in rolling_stats:
            if stat in df.columns:
                col_name = f'{stat}_rolling_{window}'
                # Calculate rolling mean using transform() with shift INSIDE the group
                # This ensures shift only happens within each player's data
                df[col_name] = (
                    df.groupby('name')[stat]
                    .transform(lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))
                )
    
    return df

## Feature Engineering: Additional Context Features
Add home/away indicators, season progression, and price momentum features.

In [ ]:
def add_context_features(df):
    """
    Add contextual features like home/away, season progression, and price changes.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added context features
    """
    
    # 1. Home/Away indicator
    df['is_home'] = df['was_home'].astype(int)
    
    # 2. Season progression features
    # Season stage: early (GW 1-13), mid (GW 14-26), late (GW 27-38)
    df['season_stage'] = pd.cut(df['GW'], bins=[0, 13, 26, 38], 
                                 labels=['early', 'mid', 'late'])
    
    # Gameweek as percentage of season completion
    df['season_progress'] = df['GW'] / 38.0
    
    # 3. Price change features (if value column exists)
    if 'value' in df.columns:
        # Sort by player and time
        df = df.sort_values(['element', 'season', 'GW']).reset_index(drop=True)
        
        # Price change from previous gameweek
        df['price_change'] = df.groupby(['element', 'season'])['value'].diff()
        
        # Cumulative price change within season (from starting price)
        df['price_change_cumulative'] = df.groupby(['element', 'season'])['value'].transform(
            lambda x: x - x.iloc[0] if len(x) > 0 else 0
        )
        
        # Price trend: increasing (1), stable (0), decreasing (-1)
        df['price_trend'] = df['price_change'].apply(
            lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
        )
    return df

### Apply All Feature Engineering Functions
Combine all feature engineering steps to create the complete dataset.

In [ ]:
# Apply rolling averages for player form
print("Adding rolling averages...")
all_seasons_data_with_rolling = add_rolling_player_stats(
    all_seasons_data_with_opponent, 
    windows=[3, 5, 10]
)

# Apply context features
print("Adding context features...")
all_seasons_data_featured = add_context_features(all_seasons_data_with_rolling)

# Check final shape
print(f"\nFinal dataset shape: {all_seasons_data_featured.shape}")
print(f"Original dataset shape: {all_seasons_data.shape}")
print(f"Total new features added: {all_seasons_data_featured.shape[1] - all_seasons_data.shape[1]}")

# verify existing features

In [ ]:
## save the final dataset with features
all_seasons_data_featured.to_csv('all_seasons_data_featured.csv', index=False)

# consider here making some cleaning

# here I am assuming that the work with data is done we start standarizing and training

In [1]:
import pandas as pd
all_seasons_data_featured = pd.read_csv('all_seasons_data_featured.csv')

In [2]:
all_seasons_data_featured.columns.tolist()
print(" all columns in the final dataset:")
print(all_seasons_data_featured.columns.tolist())
# display  the non numeric columns only
non_numeric_cols = [col for col in all_seasons_data_featured.columns if not pd.api.types.is_numeric_dtype(all_seasons_data_featured[col])]
print("Non-numeric columns in the final dataset:")
print(non_numeric_cols)

 all columns in the final dataset:
['name', 'assists', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'creativity', 'element', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackles', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW', 'position', 'team', 'defensive_contribution', 'season', 'game_number', 'assists_prev_1', 'assists_prev_2', 'assists_prev_3', 'assists_prev_4', 'assists_prev_5', 'bonus_prev_1', 'bonus_prev_2', 'bonus_prev_3', 'bonus_prev_4', 'bonus_prev_5', 'bps_prev_1', 'bps_prev_2', 'bps_prev_3', 'bps_prev_4', 'bps_prev_5', 'clean_sheets_prev_1', 'clean_sheets_prev_2', 'clean_sheets_prev_3', 'clean_sheets_prev_4', 'clean_sheets_prev_5', 'clearances_blocks_interceptions_prev

In [3]:
# Get column names
print("Total columns:", len(all_seasons_data_featured.columns))
print(all_seasons_data_featured.columns.tolist())

# Get non-numeric columns
non_numeric_cols = [col for col in all_seasons_data_featured.columns if not pd.api.types.is_numeric_dtype(all_seasons_data_featured[col])]
print("\n\nNon-numeric columns:", non_numeric_cols)

Total columns: 259
['name', 'assists', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'creativity', 'element', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackles', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW', 'position', 'team', 'defensive_contribution', 'season', 'game_number', 'assists_prev_1', 'assists_prev_2', 'assists_prev_3', 'assists_prev_4', 'assists_prev_5', 'bonus_prev_1', 'bonus_prev_2', 'bonus_prev_3', 'bonus_prev_4', 'bonus_prev_5', 'bps_prev_1', 'bps_prev_2', 'bps_prev_3', 'bps_prev_4', 'bps_prev_5', 'clean_sheets_prev_1', 'clean_sheets_prev_2', 'clean_sheets_prev_3', 'clean_sheets_prev_4', 'clean_sheets_prev_5', 'clearances_blocks_interceptions_prev_1', 'clearances

## Data Preprocessing for Regression Models
We'll prepare all features for predicting player total_points, ensuring we only use data from previous gameweeks.

In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Create a copy to work with
df = all_seasons_data_featured.copy()

print(f"Initial dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")

Initial dataset shape: (216537, 259)
Total missing values: 2513089


In [5]:
# the number of rows with missing values
rows_with_missing = df.isnull().any(axis=1).sum()
print(f"Rows with missing values: {rows_with_missing}")

Rows with missing values: 46618


### Step 1: Identify Features to Exclude from Training
These are the features that represent current or future gameweek data (not available at prediction time)

In [6]:
# Columns to exclude from features (these are current gameweek stats or identifiers)
# These features contain information about the CURRENT gameweek, not previous ones
exclude_from_training = [
    'total_points',  # Target variable
    'name',  # Identifier
    'element',  # Player ID
    'fixture',  # Match ID
    'kickoff_time',  # Time info
    'round',  # Round number
    'GW',  # Gameweek number
    'game_number',  # Game number
    'match_number',  # Match number
    
    # Current gameweek performance stats (not available at prediction time)
    'assists',  # Current GW assists
    'bonus',  # Current GW bonus
    'bps',  # Current GW BPS
    'clean_sheets',  # Current GW clean sheets
    'clearances_blocks_interceptions',  # Current GW defensive stats
    'creativity',  # Current GW creativity
    'goals_conceded',  # Current GW goals conceded
    'goals_scored',  # Current GW goals scored
    'ict_index',  # Current GW ICT
    'influence',  # Current GW influence
    'minutes',  # Current GW minutes played
    'own_goals',  # Current GW own goals
    'penalties_missed',  # Current GW penalties missed
    'penalties_saved',  # Current GW penalties saved
    'recoveries',  # Current GW recoveries
    'red_cards',  # Current GW red cards
    'saves',  # Current GW saves
    'selected',  # Current GW selection
    'tackles',  # Current GW tackles
    'team_a_score',  # Current GW away score
    'team_h_score',  # Current GW home score
    'threat',  # Current GW threat
    'transfers_balance',  # Current GW transfers
    'transfers_in',  # Current GW transfers in
    'transfers_out',  # Current GW transfers out
    'yellow_cards',  # Current GW yellow cards
    'defensive_contribution',  # Current GW defensive contribution
]

# Verify all these columns exist
exclude_from_training = [col for col in exclude_from_training if col in df.columns]

print(f"Excluding {len(exclude_from_training)} columns that represent current gameweek data:")
print(exclude_from_training)

Excluding 35 columns that represent current gameweek data:
['total_points', 'name', 'element', 'fixture', 'kickoff_time', 'round', 'GW', 'game_number', 'assists', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'creativity', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'saves', 'selected', 'tackles', 'team_a_score', 'team_h_score', 'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'yellow_cards', 'defensive_contribution']


### Step 2: Handle Non-Numerical Features
Convert all categorical/text features to numerical representations

In [7]:
# Get non-numeric columns (excluding those already in exclude list)
non_numeric_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
print("Non-numeric columns found:")
print(non_numeric_cols)

# Create encoders dictionary to store all label encoders
label_encoders = {}

# Encode categorical features
for col in non_numeric_cols:
    if col not in exclude_from_training:
        print(f"\nEncoding {col}...")
        le = LabelEncoder()
        # Handle NaN values
        df[col] = df[col].fillna('missing')
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"  - Created {col}_encoded with {len(le.classes_)} unique values")
        
# For was_home (boolean), convert to int if not already
if 'was_home' in df.columns and df['was_home'].dtype == 'bool':
    df['was_home'] = df['was_home'].astype(int)
    
# For is_home (boolean), convert to int if not already  
if 'is_home' in df.columns and df['is_home'].dtype == 'bool':
    df['is_home'] = df['is_home'].astype(int)

# Handle was_home_prev columns (convert True/False strings to 0/1)
was_home_prev_cols = [col for col in df.columns if 'was_home_prev_' in col]
for col in was_home_prev_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0})
        df[col] = df[col].fillna(0).astype(float)
        
print(f"\nEncoding complete. Created {len(label_encoders)} encoded features.")

Non-numeric columns found:
['name', 'kickoff_time', 'opponent_team', 'position', 'team', 'season', 'was_home_prev_1', 'was_home_prev_2', 'was_home_prev_3', 'was_home_prev_4', 'was_home_prev_5', 'season_stage']

Encoding opponent_team...
  - Created opponent_team_encoded with 53 unique values

Encoding position...
  - Created position_encoded with 5 unique values

Encoding team...
  - Created team_encoded with 34 unique values

Encoding season...
  - Created season_encoded with 10 unique values

Encoding was_home_prev_1...
  - Created was_home_prev_1_encoded with 3 unique values

Encoding was_home_prev_2...
  - Created was_home_prev_2_encoded with 3 unique values

Encoding was_home_prev_3...
  - Created was_home_prev_3_encoded with 3 unique values

Encoding was_home_prev_4...
  - Created was_home_prev_4_encoded with 3 unique values

Encoding was_home_prev_5...
  - Created was_home_prev_5_encoded with 3 unique values

Encoding season_stage...
  - Created season_stage_encoded with 4 uniqu

### Step 3: Select Training Features
Select only features that use previous gameweek data

In [8]:
# Get all potential feature columns (numeric only)
# This includes all columns except those explicitly excluded
all_possible_features = [col for col in df.columns if col not in exclude_from_training]

# Keep only numeric columns (includes our newly encoded features)
training_features = []
for col in all_possible_features:
    if pd.api.types.is_numeric_dtype(df[col]):
        training_features.append(col)
    elif col not in non_numeric_cols:  # Skip original non-numeric columns (we have encoded versions)
        training_features.append(col)

# Remove the original non-numeric columns from training features (keep encoded versions)
original_non_numeric = [col for col in non_numeric_cols if col not in exclude_from_training]
training_features = [col for col in training_features if col not in original_non_numeric]

print(f"Total training features: {len(training_features)}")
print(f"\nFeature categories:")

# Categorize features for better understanding
prev_features = [col for col in training_features if '_prev_' in col]
rolling_features = [col for col in training_features if '_rolling_' in col]
opponent_features = [col for col in training_features if 'opponent_' in col]
encoded_features = [col for col in training_features if '_encoded' in col]
strength_features = [col for col in training_features if 'strength' in col or 'advantage' in col]
other_features = [col for col in training_features if col not in prev_features + rolling_features + 
                  opponent_features + encoded_features + strength_features]

print(f"  - Previous gameweek features (_prev_): {len(prev_features)}")
print(f"  - Rolling average features (_rolling_): {len(rolling_features)}")
print(f"  - Opponent features: {len(opponent_features)}")
print(f"  - Strength/advantage features: {len(strength_features)}")
print(f"  - Encoded categorical features: {len(encoded_features)}")
print(f"  - Other features: {len(other_features)}")

print(f"\nOther features include: {other_features[:20]}")  # Show first 20

Total training features: 224

Feature categories:
  - Previous gameweek features (_prev_): 150
  - Rolling average features (_rolling_): 52
  - Opponent features: 13
  - Strength/advantage features: 9
  - Encoded categorical features: 10
  - Other features: 7

Other features include: ['value', 'was_home', 'is_home', 'season_progress', 'price_change', 'price_change_cumulative', 'price_trend']


### Step 4: Handle Missing Values
Fill missing values with appropriate strategies for each feature type

In [9]:
# Check missing values in training features
print("Missing values in training features:")
missing_counts = df[training_features].isnull().sum()
features_with_missing = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(f"\nFeatures with missing values: {len(features_with_missing)}")
if len(features_with_missing) > 0:
    print("\nTop 20 features with most missing values:")
    print(features_with_missing.head(20))

# Drop rows with game_number < 5
print(f"\nOriginal dataset shape: {df.shape}")
if 'game_number' in df.columns:
    rows_before = len(df)
    df = df[df['game_number'] >= 5]
    rows_dropped = rows_before - len(df)
    print(f"Dropped {rows_dropped} rows with game_number < 5")
    print(f"Dataset shape after filtering game_number >= 5: {df.shape}")
else:
    print("Warning: 'game_number' column not found in dataframe")

# Drop rows with any NaN/null values in training features or target
print("\nDropping rows with missing values...")
rows_before = len(df)
df = df.dropna(subset=training_features + ['total_points'])
rows_dropped = rows_before - len(df)
print(f"Dropped {rows_dropped} rows with missing values")

print(f"\nAfter cleaning:")
print(f"  Dataset shape: {df.shape}")
print(f"  Missing values in features: {df[training_features].isnull().sum().sum()}")
print(f"  Missing values in target: {df['total_points'].isnull().sum()}")

Missing values in training features:

Features with missing values: 208

Top 20 features with most missing values:
offensive_advantage                       25777
opponent_defensive_strength               25777
opponent_overall_team_strength            25777
opponent_offensive_strength               25777
opponent_team_goals_scored_rolling_5      25777
opponent_difficulty                       25777
overall_advantage                         25777
opponent_team_goals_scored_rolling_3      25777
opponent_team_clean_sheet_rolling_3       25777
opponent_team_total_points_rolling_5      25777
opponent_team_clean_sheet_rolling_5       25777
opponent_team_goals_conceded_rolling_5    25777
opponent_team_goals_conceded_rolling_3    25777
opponent_team_total_points_rolling_3      25777
defensive_advantage                       25777
team_h_score_prev_5                       20899
team_a_score_prev_5                       20899
assists_prev_5                            20840
bps_prev_5           

### Step 5: Prepare Features and Target
Create X (features) and y (target) datasets

In [11]:
import numpy as np
# Prepare feature matrix X and target vector y
X = df[training_features].copy()
y = df['total_points'].copy()

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nTarget variable statistics:")
print(y.describe())

# Check for any remaining infinite values
print(f"\nInfinite values in X: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

# Replace any infinite values with NaN then fill with median
if np.isinf(X.select_dtypes(include=[np.number])).sum().sum() > 0:
    print("Replacing infinite values with median...")
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].median())

print(f"\nFinal feature check:")
print(f"  X shape: {X.shape}")
print(f"  X missing values: {X.isnull().sum().sum()}")
print(f"  X infinite values: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

Feature matrix X shape: (169749, 224)
Target vector y shape: (169749,)

Target variable statistics:
count    169749.000000
mean          1.362971
std           2.536086
min          -6.000000
25%           0.000000
50%           0.000000
75%           2.000000
max          29.000000
Name: total_points, dtype: float64

Infinite values in X: 0

Final feature check:
  X shape: (169749, 224)
  X missing values: 0
  X infinite values: 0


### Step 6: Standardize Features
Apply StandardScaler to normalize all features to the same range

In [12]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform the features
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame to maintain column names
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Feature standardization complete!")
print(f"\nOriginal features sample statistics (before scaling):")
print(X.describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print(f"\nScaled features sample statistics (after scaling):")
print(X_scaled.describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print(f"\nScaled features shape: {X_scaled.shape}")
print(f"All features now have mean ≈ 0 and std ≈ 1")

Feature standardization complete!

Original features sample statistics (before scaling):
           value  was_home  assists_prev_1  assists_prev_2  assists_prev_3
mean   50.657642  0.500068        0.038033        0.038209        0.038398
std    12.086296  0.500001        0.207496        0.208031        0.208477
min    36.000000  0.000000        0.000000        0.000000        0.000000
max   145.000000  1.000000        4.000000        4.000000        4.000000

Scaled features sample statistics (after scaling):
             value      was_home  assists_prev_1  assists_prev_2  \
mean -2.143152e-16  1.737125e-18    5.023012e-17   -7.300111e-17   
std   1.000003e+00  1.000003e+00    1.000003e+00    1.000003e+00   
min  -1.212752e+00 -1.000136e+00   -1.832934e-01   -1.836719e-01   
max   7.805752e+00  9.998645e-01    1.909420e+01    1.904428e+01   

      assists_prev_3  
mean   -6.898270e-17  
std     1.000003e+00  
min    -1.841830e-01  
max     1.900261e+01  

Scaled features shape: (169

### Step 7: Train-Test Split
Split data into training and testing sets (80-20 split)

In [13]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42,
    shuffle=True
)

print("Data split complete!")
print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  y_train mean: {y_train.mean():.3f}")
print(f"  y_train std: {y_train.std():.3f}")

print(f"\nTesting set:")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")
print(f"  y_test mean: {y_test.mean():.3f}")
print(f"  y_test std: {y_test.std():.3f}")

print(f"\nTotal samples: {len(X_scaled)}")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X_scaled)*100:.1f}%)")

Data split complete!

Training set:
  X_train shape: (135799, 224)
  y_train shape: (135799,)
  y_train mean: 1.364
  y_train std: 2.538

Testing set:
  X_test shape: (33950, 224)
  y_test shape: (33950,)
  y_test mean: 1.359
  y_test std: 2.527

Total samples: 169749
Training samples: 135799 (80.0%)
Testing samples: 33950 (20.0%)


### Summary of Data Preprocessing
Review what we've done

In [14]:
print("="*70)
print("DATA PREPROCESSING SUMMARY")
print("="*70)

print("\n1. FEATURES USED:")
print(f"   - Total features: {len(training_features)}")
print(f"   - Previous gameweek stats (_prev_): {len(prev_features)}")
print(f"   - Rolling averages (_rolling_): {len(rolling_features)}")
print(f"   - Opponent features: {len(opponent_features)}")
print(f"   - Strength/advantage features: {len(strength_features)}")
print(f"   - Encoded categorical features: {len(encoded_features)}")
print(f"   - Other predictive features: {len(other_features)}")

print("\n2. FEATURES EXCLUDED (current gameweek data):")
print(f"   - {len(exclude_from_training)} features excluded")
print(f"   - These represent CURRENT gameweek performance")
print(f"   - Not available at prediction time")

print("\n3. DATA TRANSFORMATIONS:")
print(f"   ✓ Non-numeric features encoded to numeric")
print(f"   ✓ Missing values handled appropriately")
print(f"   ✓ All features standardized (mean=0, std=1)")
print(f"   ✓ No infinite values")
print(f"   ✓ Boolean features converted to 0/1")

print("\n4. DATASET SPLIT:")
print(f"   - Total samples: {len(X_scaled):,}")
print(f"   - Training: {len(X_train):,} samples ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"   - Testing: {len(X_test):,} samples ({len(X_test)/len(X_scaled)*100:.1f}%)")

print("\n5. TARGET VARIABLE (total_points):")
print(f"   - Mean: {y.mean():.3f}")
print(f"   - Std: {y.std():.3f}")
print(f"   - Min: {y.min():.0f}")
print(f"   - Max: {y.max():.0f}")

print("\n6. KEY PRINCIPLE:")
print("   ⚠ All training features use ONLY previous gameweek data")
print("   ⚠ No data leakage from current/future gameweeks")
print("   ⚠ Model will predict future performance based on past")

print("\n" + "="*70)
print("READY FOR MODEL TRAINING!")
print("="*70)

DATA PREPROCESSING SUMMARY

1. FEATURES USED:
   - Total features: 224
   - Previous gameweek stats (_prev_): 150
   - Rolling averages (_rolling_): 52
   - Opponent features: 13
   - Strength/advantage features: 9
   - Encoded categorical features: 10
   - Other predictive features: 7

2. FEATURES EXCLUDED (current gameweek data):
   - 35 features excluded
   - These represent CURRENT gameweek performance
   - Not available at prediction time

3. DATA TRANSFORMATIONS:
   ✓ Non-numeric features encoded to numeric
   ✓ Missing values handled appropriately
   ✓ All features standardized (mean=0, std=1)
   ✓ No infinite values
   ✓ Boolean features converted to 0/1

4. DATASET SPLIT:
   - Total samples: 169,749
   - Training: 135,799 samples (80.0%)
   - Testing: 33,950 samples (20.0%)

5. TARGET VARIABLE (total_points):
   - Mean: 1.363
   - Std: 2.536
   - Min: -6
   - Max: 29

6. KEY PRINCIPLE:
   ⚠ All training features use ONLY previous gameweek data
   ⚠ No data leakage from current

## Regression Model Training
Train multiple regression models to predict player total_points

In [15]:
# Import regression models and evaluation metrics
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

print("All regression libraries imported successfully!")

All regression libraries imported successfully!


In [17]:
from collections import defaultdict

def auto_group_features(columns):
    groups = defaultdict(list)

    for col in columns:
        base = col.split("_prev")[0]
        base = base.split("_rolling")[0]
        groups[base].append(col)

    return dict(groups)
# Group features based on common prefixes
feature_groups = auto_group_features(training_features)
print("Feature groups identified:")
for group, cols in feature_groups.items():
    print(f"  - {group}: {len(cols)} features")
# print the keys as list 
print("Feature group keys:", list(feature_groups.keys()))
    

Feature groups identified:
  - value: 6 features
  - was_home: 6 features
  - assists: 8 features
  - bonus: 8 features
  - bps: 8 features
  - clean_sheets: 8 features
  - clearances_blocks_interceptions: 5 features
  - creativity: 8 features
  - goals_conceded: 5 features
  - goals_scored: 8 features
  - ict_index: 8 features
  - influence: 8 features
  - minutes: 8 features
  - own_goals: 5 features
  - penalties_missed: 5 features
  - penalties_saved: 5 features
  - recoveries: 5 features
  - red_cards: 5 features
  - saves: 8 features
  - selected: 5 features
  - tackles: 5 features
  - team_a_score: 5 features
  - team_h_score: 5 features
  - threat: 8 features
  - total_points: 8 features
  - transfers_balance: 5 features
  - transfers_in: 5 features
  - transfers_out: 5 features
  - yellow_cards: 5 features
  - defensive_contribution: 5 features
  - defensive_strength: 1 features
  - offensive_strength: 1 features
  - overall_team_strength: 1 features
  - team_goals_scored: 2 f

In [18]:

COMMON_FEATURES  = feature_groups.get('value', []) + \
    feature_groups.get('bonus', []) + \
    feature_groups.get('bps', []) + \
    feature_groups.get('clearances_blocks_interceptions', []) + \
    feature_groups.get('minutes', []) + \
    feature_groups.get('own_goals', []) + \
    feature_groups.get('red_cards', []) + \
    feature_groups.get('selected', []) + \
    feature_groups.get('total_points', []) + \
    feature_groups.get('transfers_balance', []) + \
    feature_groups.get('transfers_in', []) + \
    feature_groups.get('transfers_out', []) + \
    feature_groups.get('yellow_cards', []) + \
    feature_groups.get('overall_team_strength', []) + \
    feature_groups.get('team_total_points', []) + \
    feature_groups.get('opponent_overall_team_strength', []) + \
    feature_groups.get('opponent_team_total_points', []) + \
    feature_groups.get('offensive_advantage', []) + \
    feature_groups.get('defensive_advantage', []) + \
    feature_groups.get('overall_advantage', []) + \
    feature_groups.get('opponent_difficulty', []) + \
    feature_groups.get('is_home', []) + \
    feature_groups.get('price_change', []) + \
    feature_groups.get('price_change_cumulative', []) + \
    feature_groups.get('price_trend', [])
    
    


ONLY_NOT_GK_FEATURES = feature_groups.get('asssists', []) + \
    feature_groups.get('creativity', []) + \
    feature_groups.get('goals_scored', []) + \
    feature_groups.get('ict_index', []) + \
    feature_groups.get('influence', []) + \
    feature_groups.get('penalties_missed', []) + \
    feature_groups.get('recoveries', []) + \
    feature_groups.get('tackles', []) + \
    feature_groups.get('threat', []) + \
    feature_groups.get('defensive_contribution', []) + \
    feature_groups.get('offensive_strength', []) + \
    feature_groups.get('team_goals_scored', []) + \
    feature_groups.get('opponent_defensive_strength', []) + \
    feature_groups.get('opponent_team_goals_conceded', [])



ONLY_NOT_FWD_FEATURES = feature_groups.get('clean_sheets', []) + \
    feature_groups.get('goals_conceded', []) + \
    feature_groups.get('goals_conceded', []) + \
    feature_groups.get('defensive_strength', []) + \
    feature_groups.get('team_goals_conceded', []) + \
    feature_groups.get('team_clean_sheet', []) + \
    feature_groups.get('opponent_offensive_strength', []) + \
    feature_groups.get('opponent_team_goals_scored', []) + \
    feature_groups.get('opponent_team_clean_sheet', [])
    

ONLY_GPK_FEATURES = feature_groups.get('saves', []) + \
    feature_groups.get('penalties_saved', []) + \
    feature_groups.get('team_saves', [])

GK_FEATURES = COMMON_FEATURES + ONLY_GPK_FEATURES + ONLY_NOT_FWD_FEATURES
DEF_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
MID_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
FWD_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES



POSITION_FEATURES = {
    'GK': GK_FEATURES,
    'DEF': DEF_FEATURES,
    'MID': MID_FEATURES,
    'FWD': FWD_FEATURES
}

print("Position-specific feature sets defined!")
for pos, features in POSITION_FEATURES.items():
    print(f"{pos}: {len(features)} features")

Position-specific feature sets defined!
GK: 133 features
DEF: 186 features
MID: 186 features
FWD: 186 features


## preparing data for each position

In [ ]:
# Prepare data for modeling
def prepare_position_data(df, position, features):
    """Prepare data for a specific position"""
    
    # Filter by position 
    pos_df = df[df['position'] == position].copy()
    
    # Get available features (some may not exist)
    available_features = [f for f in features if f in pos_df.columns]
    
    # Remove rows with missing target
    pos_df = pos_df.dropna(subset=['total_points'])
    
    # Fill missing features with 0
    for col in available_features:
        pos_df[col] = pos_df[col].fillna(0)
    
    # Remove infinite values
    pos_df = pos_df.replace([np.inf, -np.inf], 0)
    
    X = pos_df[available_features]
    y = pos_df['total_points']
    
    return X, y, available_features, pos_df

print("Data preparation function defined!")

## 4. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define hyperparameter grids for different models
PARAM_GRIDS = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 9],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100]
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1],
        'l1_ratio': [0.2, 0.5, 0.8]
    }
}

PARAM_GRIDS['XGBoost'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

PARAM_GRIDS['LightGBM'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9, 1.0]
}

print("Hyperparameter grids defined for models:")
for model_name in PARAM_GRIDS:
    print(f"  - {model_name}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
def tune_model(model, param_grid, X_train, y_train, cv=5):
    """Perform GridSearchCV for hyperparameter tuning"""
    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def tune_model_randomized(model, param_distributions, X_train, y_train, n_iter=50, cv=5):
    """Perform RandomizedSearchCV for faster hyperparameter tuning"""
    
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    random_search.fit(X_train, y_train)
    
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

print("Hyperparameter tuning functions defined!")

## 5. Import Additional Libraries for Model Training

In [ ]:
# Additional imports for model training
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
import xgboost as xgb
import lightgbm as lgb

print("Additional libraries imported successfully!")

## 6. Train/Validation/Test Split and Model Training Pipeline

In [ ]:
# Define the models to train
MODELS = {
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet(max_iter=10000),
    #'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    #'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    #'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

# Reduced parameter grids for faster training (use full grids for production)
PARAM_GRIDS_REDUCED = {
    'Ridge': {'alpha': [0.1, 1, 10]},
    'ElasticNet': {'alpha': [0.1, 1], 'l1_ratio': [0.5, 0.8]},
    #'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20], 'min_samples_split': [2, 5]},
    #'GradientBoosting': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    'XGBoost': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    #'LightGBM': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]}
}

print("Models and reduced parameter grids defined!")
print(f"Models to train: {list(MODELS.keys())}")

In [ ]:
def train_and_evaluate_models(df, positions, position_features, models, param_grids, use_tuning=False):
    """
    Train and evaluate multiple models for each position.
    
    Parameters:
    - df: DataFrame with all data
    - positions: List of positions to train models for
    - position_features: Dictionary mapping positions to their features
    - models: Dictionary of model instances
    - param_grids: Dictionary of parameter grids for tuning
    - use_tuning: Whether to perform hyperparameter tuning
    
    Returns:
    - results: Dictionary with all results
    """
    
    results = {}
    
    for position in positions:
        print(f"\n{'='*80}")
        print(f"Training models for position: {position}")
        print(f"{'='*80}")
        
        # Prepare data for this position
        features = position_features[position]
        X, y, available_features, pos_df = prepare_position_data(df, position, features)
        
        print(f"Data shape: X={X.shape}, y={y.shape}")
        print(f"Available features: {len(available_features)}")
        
        if len(X) < 100:
            print(f"Skipping {position} - not enough data")
            continue
        
        # Split data: 70% train, 15% validation, 15% test
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=0.15, random_state=42
        )
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=0.176, random_state=42  # 0.176 of 85% ≈ 15%
        )
        
        print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}, Test size: {len(X_test)}")
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)
        
        position_results = {
            'scaler': scaler,
            'features': available_features,
            'X_test': X_test,
            'y_test': y_test,
            'models': {}
        }
        
        # Train each model
        for model_name, model in models.items():
            print(f"\n--- Training {model_name} ---")
            start_time = time.time()
            
            try:
                # Clone the model to avoid issues with refitting
                model_clone = type(model)(**model.get_params())
                
                if use_tuning and model_name in param_grids:
                    # Perform hyperparameter tuning
                    print(f"Tuning hyperparameters...")
                    grid_search = GridSearchCV(
                        model_clone, param_grids[model_name],
                        cv=3, scoring='neg_mean_squared_error', n_jobs=-1
                    )
                    grid_search.fit(X_train_scaled, y_train)
                    best_model = grid_search.best_estimator_
                    best_params = grid_search.best_params_
                    print(f"Best params: {best_params}")
                else:
                    # Train without tuning
                    best_model = model_clone
                    best_model.fit(X_train_scaled, y_train)
                    best_params = None
                
                # Predictions
                y_train_pred = best_model.predict(X_train_scaled)
                y_val_pred = best_model.predict(X_val_scaled)
                y_test_pred = best_model.predict(X_test_scaled)
                
                # Calculate metrics
                train_metrics = {
                    'r2': r2_score(y_train, y_train_pred),
                    'mae': mean_absolute_error(y_train, y_train_pred),
                    'mse': mean_squared_error(y_train, y_train_pred),
                    'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred))
                }
                
                val_metrics = {
                    'r2': r2_score(y_val, y_val_pred),
                    'mae': mean_absolute_error(y_val, y_val_pred),
                    'mse': mean_squared_error(y_val, y_val_pred),
                    'rmse': np.sqrt(mean_squared_error(y_val, y_val_pred))
                }
                
                test_metrics = {
                    'r2': r2_score(y_test, y_test_pred),
                    'mae': mean_absolute_error(y_test, y_test_pred),
                    'mse': mean_squared_error(y_test, y_test_pred),
                    'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
                }
                
                training_time = time.time() - start_time
                
                position_results['models'][model_name] = {
                    'model': best_model,
                    'best_params': best_params,
                    'train_metrics': train_metrics,
                    'val_metrics': val_metrics,
                    'test_metrics': test_metrics,
                    'training_time': training_time,
                    'y_test_pred': y_test_pred
                }
                
                print(f"Training R²: {train_metrics['r2']:.4f}, Validation R²: {val_metrics['r2']:.4f}, Test R²: {test_metrics['r2']:.4f}")
                print(f"Test MAE: {test_metrics['mae']:.4f}, Test RMSE: {test_metrics['rmse']:.4f}")
                print(f"Training time: {training_time:.2f}s")
                
            except Exception as e:
                print(f"Error training {model_name}: {str(e)}")
                continue
        
        results[position] = position_results
    
    return results

print("Training and evaluation function defined!")

## 7. Train Models for All Positions

In [ ]:
# Train models for all positions
# Set use_tuning=True for hyperparameter tuning (slower but potentially better results)
# Set use_tuning=False for faster training with default parameters

positions_to_train = ['GK', 'DEF', 'MID', 'FWD']

print("Starting model training for all positions...")
print("This may take a few minutes...\n")

# Train with reduced parameter grid for faster execution
all_results = train_and_evaluate_models(
    df=all_seasons_data,  # Use the main dataframe
    positions=positions_to_train,
    position_features=POSITION_FEATURES,
    models=MODELS,
    param_grids=PARAM_GRIDS_REDUCED,
    use_tuning=True  # Set to True for hyperparameter tuning
)

print("\n" + "="*80)
print("Model training completed for all positions!")
print("="*80)

## 8. Model Comparison and Results Summary

In [ ]:
def create_results_summary(results):
    """Create a summary DataFrame of all model results"""
    
    summary_data = []
    
    for position, pos_results in results.items():
        for model_name, model_results in pos_results['models'].items():
            summary_data.append({
                'Position': position,
                'Model': model_name,
                'Train_R2': model_results['train_metrics']['r2'],
                'Val_R2': model_results['val_metrics']['r2'],
                'Test_R2': model_results['test_metrics']['r2'],
                'Train_MAE': model_results['train_metrics']['mae'],
                'Val_MAE': model_results['val_metrics']['mae'],
                'Test_MAE': model_results['test_metrics']['mae'],
                'Train_RMSE': model_results['train_metrics']['rmse'],
                'Val_RMSE': model_results['val_metrics']['rmse'],
                'Test_RMSE': model_results['test_metrics']['rmse'],
                'Training_Time': model_results['training_time']
            })
    
    summary_df = pd.DataFrame(summary_data)
    return summary_df

# Create and display summary
summary_df = create_results_summary(all_results)
print("="*100)
print("MODEL PERFORMANCE SUMMARY - ALL POSITIONS")
print("="*100)
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('model_results_summary.csv', index=False)
print("\nResults saved to 'model_results_summary.csv'")

In [ ]:
# Display best model for each position
print("\n" + "="*80)
print("BEST MODEL FOR EACH POSITION (Based on Test R²)")
print("="*80)

for position in positions_to_train:
    if position in all_results:
        pos_models = all_results[position]['models']
        if pos_models:
            best_model_name = max(pos_models.keys(), key=lambda x: pos_models[x]['test_metrics']['r2'])
            best_r2 = pos_models[best_model_name]['test_metrics']['r2']
            best_mae = pos_models[best_model_name]['test_metrics']['mae']
            best_rmse = pos_models[best_model_name]['test_metrics']['rmse']
            print(f"\n{position}:")
            print(f"  Best Model: {best_model_name}")
            print(f"  Test R²: {best_r2:.4f}")
            print(f"  Test MAE: {best_mae:.4f}")
            print(f"  Test RMSE: {best_rmse:.4f}")

## 9. Visualize Model Comparison by Position

In [ ]:
# Plot R² scores comparison for each position
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, position in enumerate(positions_to_train):
    if position in all_results:
        pos_models = all_results[position]['models']
        model_names = list(pos_models.keys())
        train_r2 = [pos_models[m]['train_metrics']['r2'] for m in model_names]
        val_r2 = [pos_models[m]['val_metrics']['r2'] for m in model_names]
        test_r2 = [pos_models[m]['test_metrics']['r2'] for m in model_names]
        
        x = np.arange(len(model_names))
        width = 0.25
        
        axes[idx].bar(x - width, train_r2, width, label='Train R²', color='steelblue', alpha=0.8)
        axes[idx].bar(x, val_r2, width, label='Validation R²', color='orange', alpha=0.8)
        axes[idx].bar(x + width, test_r2, width, label='Test R²', color='green', alpha=0.8)
        
        axes[idx].set_xlabel('Model')
        axes[idx].set_ylabel('R² Score')
        axes[idx].set_title(f'{position} - R² Score Comparison')
        axes[idx].set_xticks(x)
        axes[idx].set_xticklabels(model_names, rotation=45, ha='right')
        axes[idx].legend()
        axes[idx].set_ylim([0, 1])
        axes[idx].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        
        # Add value labels on bars
        for i, (tr, vr, ter) in enumerate(zip(train_r2, val_r2, test_r2)):
            axes[idx].text(i - width, tr + 0.02, f'{tr:.2f}', ha='center', va='bottom', fontsize=7)
            axes[idx].text(i, vr + 0.02, f'{vr:.2f}', ha='center', va='bottom', fontsize=7)
            axes[idx].text(i + width, ter + 0.02, f'{ter:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('model_r2_comparison_by_position.png', dpi=150, bbox_inches='tight')
plt.show()
print("R² comparison chart saved to 'model_r2_comparison_by_position.png'")